<a href="https://colab.research.google.com/github/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System/blob/main/EDA_MULTI_SESSION_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exploratory Data Analysis: across sessions**

# Categorization of Lengths in the Rhesus Monkey (Macaca Mulatta) with a Three-Category System

Paola Castillo

This notebook pools **several sessions**: learning curves, per-session
psychometric parameters, mixed-effects models, the per-trial features of
every session (pairplots, violins, heatmaps and the per-feature statistics,
session by session), the timing measures including the movement takeoff
(2.6), and PCA / t-SNE embeddings coloured by category, outcome and session
(2.7). Sessions are always ordered
by the date parsed from their runTag (`sessROM_<dd-mmm-yyyy>_<HH-MM>`).
Section numbers keep the ones used in the single-session notebook (section 1
there, section 2 here, section 3 in the RNN dataset notebook), so
cross-references between them still hold:
wherever a cell below mentions section 1.x it means that notebook, and the code
those cells depend on is copied into 0.2.

Point 0.1 at the `outputs/` tree (optionally with a date window) and run
section 2 top to bottom. The RNN-ready dataset (former section 3) lives in
`RNN_DATASET_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb`. Analysing one session on its own is the job of
the companion notebook
`EDA_SINGLE_SESSION_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb`.


## 0. GitHub Repository Connection

In [ ]:
# Remove any clone left by an earlier run of this runtime. Without this,
# `git clone` below fails with "destination path already exists" and the
# notebook silently keeps reading the OLD data.
!rm -rf "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

In [ ]:
!git clone https://github.com/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System.git

Cloning into 'Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 222 (delta 108), reused 131 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 411.91 KiB | 2.92 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [ ]:
!ls "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

centerTask_v8.17  fonts  README.md


## 0.1. Loading sessions from `outputs/`

The task writes one folder per session under `outputs/`, named after the
runTag it stamps everything with
(`sessROM_<dd-mmm-yyyy>_<HH-MM>`, e.g. `sessROM_26-Aug-2026_12-28`). Each
folder holds that session's `trial_data_*.csv`, both trajectory exports,
`session_data_*.csv` and the printed `session_report_*.txt`.

This cell reads that tree directly; every session to be pooled has to live
there.

- **Filtering by date.** Set `SESSION_START_DATE` and `SESSION_END_DATE` to
  anything pandas can parse (`'2026-08-21'`, `'21-Aug-2026'`, a datetime) or
  leave them `None` for no bound. Both ends are inclusive, and a bare end
  date covers that whole day, so passing the same value twice selects one
  day's sessions. The timestamp is parsed from the runTag rather than from
  the `Date` column inside the CSV, which is what lets two sessions recorded
  on the same day still be ordered.

- **What it sets.** `SESSION_INVENTORY` is the table of sessions in the
  window with a path per file kind (`None` where a session predates that
  export). `POOLED_FROM_OUTPUTS` is every trial in the window in one frame,
  tagged with `Session` / `SessionStart` / `SessionIndex`. `SESSION_GLOB` is
  pointed at the same folders, so section 2 pools exactly this window.

- **Invalid sessions.** `INVALID_SESSIONS` lists sessions whose timing is not
  usable (the pre-v8.21 fixed-anchor clock made the RZ2 timestamps drift
  against the task clock, so decision windows shrank during the session).
  With `EXCLUDE_INVALID_SESSIONS = True` they are dropped from the inventory
  and the reason is printed; set it to `False` to keep them, flagged in the
  `InvalidReason` column. Entries match a runTag exactly or as a substring;
  an entry that matches no folder is reported, so a short name can be
  replaced by the full runTag.

Analysing one session on its own (per-trial figures, feature engineering,
PCA, clustering, the single-session psychometric fit) is the job of the
companion single-session notebook.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = "outputs"
OUTPUTS_GLOBS = ["outputs", "/content/*/outputs", "../outputs"]
SESSION_START_DATE = None
SESSION_END_DATE = None
RUNTAG_DATE_RE = re.compile(r"_(\d{2}-[A-Za-z]{3}-\d{4})_(\d{2})-(\d{2})$")
SESSION_FILE_KINDS = {
    "trial_data": "trial_data_{tag}.csv",
    "trajectory": "trajectory_{tag}.csv",
    "trajectory_movement": "trajectory_movement_{tag}.csv",
    "session_data": "session_data_{tag}.csv",
    "session_report": "session_report_{tag}.txt",
    "trial_kinematics": "trial_kinematics_{tag}.csv",
    "foil_events": "foil_events_{tag}.csv",
}

INVALID_SESSIONS = {
    "sessROM_31-Aug-2026_14-20": "pre-v8.21 fixed-anchor clock: RZ2 stamps drifted ~13.6 ms/s "
                                 "from GetSecs, so sample-anchored windows shrank",
    "sessPX-309": "pre-v8.21 fixed-anchor clock (03-Sep): 13.7 ms/s drift, windows had "
                  "already expired from trial 27 on",
}
EXCLUDE_INVALID_SESSIONS = True


def invalid_reason(run_tag, invalid=None):
    invalid = INVALID_SESSIONS if invalid is None else invalid
    tag = str(run_tag)
    for key, why in invalid.items():
        if key == tag or key in tag:
            return why
    return None


def _normalize(td):
    """normalize_trial_schema if the analysis cells have already run, else the
    same canonicalisation inline.

    Duplicated deliberately: this cell is meant to be runnable FIRST, before
    the cells that define the real one, so that the paths it sets are ready
    for them. Keeping the fallback means loading never depends on execution
    order.
    """
    fn = globals().get("normalize_trial_schema")
    if callable(fn):
        return fn(td)
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


def resolve_outputs_dir(candidates=OUTPUTS_GLOBS):
    """Find the outputs/ tree the task writes its sessions into.

    Checked in order so the same notebook works from a local checkout and
    from Colab, where the repo lands under /content/<repo>/. Returns None
    rather than raising, so the notebook still loads without it.
    """
    import glob as _g
    for cand in candidates:
        for hit in sorted(_g.glob(cand)):
            if Path(hit).is_dir():
                return Path(hit)
    return None


def session_timestamp(run_tag):
    """Session start time parsed out of the runTag.

    CenterOutTask.m builds it as datestr(now, 'dd-mmm-yyyy_HH-MM'), so
    'sessROM_26-Aug-2026_12-28' carries the real start instant. Parsing the
    tag rather than the Date column inside the CSV is what lets two sessions
    recorded on the SAME day still be ordered, and what makes the date filter
    below work without opening a single file.
    """
    m = RUNTAG_DATE_RE.search(str(run_tag))
    if not m:
        return pd.NaT
    return pd.to_datetime(f"{m.group(1)} {m.group(2)}:{m.group(3)}",
                          format="%d-%b-%Y %H:%M", errors="coerce")


def discover_sessions(outputs_dir=None, start_date=None, end_date=None, verbose=True,
                      exclude_invalid=None):
    """Inventory of every session folder under outputs/, filtered by date.

    One row per session, one column per file kind, holding the path when that
    file exists and None when it does not -- older sessions carry no
    foil_events or trial_kinematics, and nothing here should have to care.

    start_date / end_date are inclusive and accept anything pandas can parse
    ('2026-08-21', '21-Aug-2026', a datetime). end_date given as a bare date
    covers that whole day, so passing the same value for both selects one
    day's sessions rather than only those recorded at exactly midnight.
    """
    outputs_dir = Path(outputs_dir) if outputs_dir else resolve_outputs_dir()
    if outputs_dir is None or not outputs_dir.exists():
        print(f"No outputs/ directory found (looked in {OUTPUTS_GLOBS}). "
              f"Set OUTPUTS_DIR to its path and re-run.")
        return pd.DataFrame()
    rows = []
    for d in sorted(p for p in outputs_dir.iterdir() if p.is_dir()):
        tag = d.name
        rec = {"Session": tag, "SessionStart": session_timestamp(tag), "folder": str(d)}
        for kind, pattern in SESSION_FILE_KINDS.items():
            f = d / pattern.format(tag=tag)
            rec[kind] = str(f) if f.exists() else None
        if rec["trial_data"] is None:
            hits = sorted(d.glob("trial_data_*.csv"))
            rec["trial_data"] = str(hits[0]) if hits else None
        rows.append(rec)
    inv = pd.DataFrame(rows)
    if inv.empty:
        print(f"{outputs_dir} has no session sub-folders.")
        return inv
    n_all = len(inv)
    all_tags = list(inv["Session"])
    exclude_invalid = EXCLUDE_INVALID_SESSIONS if exclude_invalid is None else exclude_invalid
    if start_date is not None:
        lo = pd.to_datetime(start_date)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] >= lo)]
    if end_date is not None:
        hi = pd.to_datetime(end_date)
        if hi == hi.normalize():
            hi = hi + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] <= hi)]
    inv["InvalidReason"] = inv["Session"].map(invalid_reason)
    flagged = inv[inv["InvalidReason"].notna()]
    if exclude_invalid:
        inv = inv[inv["InvalidReason"].isna()]
    inv = inv.sort_values("SessionStart").reset_index(drop=True)
    inv.insert(0, "index", np.arange(1, len(inv) + 1))
    if verbose:
        window = ""
        if start_date is not None or end_date is not None:
            window = f"  [filter: {start_date or 'earliest'} .. {end_date or 'latest'}]"
        print(f"{outputs_dir}: {len(inv)}/{n_all} sessions{window}\n")
        show = inv[["index", "Session", "SessionStart"]].copy()
        show["files"] = [sum(r[k] is not None for k in SESSION_FILE_KINDS)
                         for _, r in inv.iterrows()]
        print(show.to_string(index=False))
        missing = [k for k in SESSION_FILE_KINDS if inv[k].isna().all()]
        if missing:
            print(f"\n  No session in this window has: {missing} "
                  f"(expected for sessions recorded before those exports existed).")
        if len(flagged):
            state = "excluded" if exclude_invalid else "kept, see the InvalidReason column"
            print(f"\nInvalid sessions in this window ({state}):")
            for _, fr in flagged.iterrows():
                print(f"  {fr['Session']}: {fr['InvalidReason']}")
        unmatched = [k for k in INVALID_SESSIONS if not any(k == s or k in s for s in all_tags)]
        if unmatched:
            print(f"\n  INVALID_SESSIONS entries that match no folder under {outputs_dir}: {unmatched}. "
                  f"If an entry is a short name, replace it with the session's full runTag.")
    return inv


def load_sessions_from_outputs(outputs_dir=None, start_date=None, end_date=None,
                               min_trials=20, verbose=True):
    """Pool the trial tables of every session in the date window into one frame.

    Each file goes through normalize_trial_schema, so a window may freely mix
    engine generations. Session identity and start time are carried on every
    row, because the across-session models need session as a grouping factor.
    """
    inv = discover_sessions(outputs_dir, start_date, end_date, verbose=verbose)
    if inv.empty:
        return pd.DataFrame(), inv
    frames, skipped = [], []
    for _, r in inv.iterrows():
        if not r["trial_data"]:
            skipped.append((r["Session"], "no trial_data_*.csv")); continue
        try:
            td = _normalize(pd.read_csv(r["trial_data"]))
        except Exception as exc:
            skipped.append((r["Session"], f"unreadable: {exc}")); continue
        if len(td) < min_trials:
            skipped.append((r["Session"], f"only {len(td)} trials")); continue
        td["Session"] = r["Session"]
        td["SessionStart"] = r["SessionStart"]
        td["SessionIndex"] = r["index"]
        td["SourceFile"] = Path(r["trial_data"]).name
        frames.append(td)
    if not frames:
        print("No usable trial tables in this window.")
        for sid, why in skipped: print(f"  skipped {sid}: {why}")
        return pd.DataFrame(), inv
    pooled = pd.concat(frames, ignore_index=True, sort=False)
    if verbose:
        print(f"\nPooled {pooled['Session'].nunique()} sessions, {len(pooled)} trials.")
        for sid, why in skipped: print(f"  skipped {sid}: {why}")
        per = (pooled.groupby(["SessionIndex", "Session"])
               .agg(trials=("IsCorrect", "size"), accuracy=("IsCorrect", "mean")).reset_index())
        print("\n" + per.round(4).to_string(index=False))
    return pooled, inv


REQUIRED_TRIAL_FIELDS = ["NumCategories", "SessionMode"]


def check_session_fields(inv, fields=REQUIRED_TRIAL_FIELDS):
    """Warn when a trial_data file lacks a field the analyses read, and say
    WHICH file was read: a NaN in NumCategories / SessionMode for a session
    that has them in the repository means the notebook is reading a stale
    copy (typically a Colab runtime whose old clone was not removed)."""
    if inv is None or inv.empty:
        return
    missing = []
    for _, r in inv.iterrows():
        if not r.get("trial_data"):
            continue
        with open(r["trial_data"]) as fh:
            header = fh.readline().strip().split(",")
        lack = [f for f in fields if f not in header]
        if lack:
            missing.append((r["Session"], lack, r["trial_data"]))
    if missing:
        print(f"\nWARNING: {len(missing)} session file(s) lack {fields}:")
        for tag, lack, path in missing:
            print(f"  {tag}: missing {lack}  <- {path}")
        print("  If the repository has these columns, the files being read are stale: "
              "re-run the clone cell in section 0 (it now removes the old clone) "
              "or pull the latest outputs/.")


SESSION_INVENTORY = discover_sessions(OUTPUTS_DIR if Path(OUTPUTS_DIR).exists() else None,
                                      SESSION_START_DATE, SESSION_END_DATE)
check_session_fields(SESSION_INVENTORY)
POOLED_FROM_OUTPUTS, _INV = load_sessions_from_outputs(
    OUTPUTS_DIR if Path(OUTPUTS_DIR).exists() else None,
    SESSION_START_DATE, SESSION_END_DATE, verbose=True)

if not SESSION_INVENTORY.empty:
    _resolved = resolve_outputs_dir() if not Path(OUTPUTS_DIR).exists() else Path(OUTPUTS_DIR)
    SESSION_GLOB = str(Path(_resolved) / "*" / "trial_data_*.csv")
    print(f"\nSESSION_GLOB set to {SESSION_GLOB!r} -- section 2 will pool the same sessions.")


## 0.2. Shared definitions

Section 2 reuses code that is defined and explained in the
single-session notebook. The three cells below carry a verbatim copy of exactly
what they need, so this notebook runs on its own:

| Cell | Copied from | Provides |
|---|---|---|
| pipeline | 1.1 Signal processing | `TrajectoryProcessor`, `TrajectoryDataset`, `normalize_trial_schema`, `TrialInfoTable`, `FIGURE_DPI`, font/figure style |
| features | 1.2 Feature engineering | `build_trial_features`, `drop_early_exits`, the category / correctness palettes, `pairplot_trials`, `violin_by_group`, `correlation_heatmap`, `correct_vs_incorrect`, `accuracy_breakdown` |
| fits | 1.5 / 1.6 / 1.8 | `fit_all_boundaries` and the psychometric helpers, `boundary_distance`, `NOMINAL_BOUNDARIES`, `PERCEPTUAL_ERRORS` / `MOTOR_ERRORS` |

Nothing here is executed on data: these cells only define functions and
constants. If the pipeline or the psychometric model changes in the
single-session notebook, update the copies here.


In [ ]:
"""Signal-processing pipeline, copied from section 1.1 of the single-session
notebook (constants, font/figure style, Hampel -> Kalman+RTS -> PCHIP ->
Butterworth, TrajectoryDataset, normalize_trial_schema, TrialInfoTable).

Only the pieces the across-session cells read are kept: the per-trial figure
drawer and run() stay in the single-session notebook. Keep the two copies in
sync when the pipeline changes.
"""

import glob
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.signal import lfilter, savgol_filter
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MultipleLocator
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

GRID_DT_S = 0.008
CUTOFF_HZ = 6.0
ACCEL_PEAK_PERCENTILE = 99.0
SAVGOL_WINDOW = 11
SAVGOL_POLYORDER = 3
MOVE_SPEED_FRAC = 0.05
MOVE_SPEED_FRAC_ALT = 0.10
PIXEL_PITCH_MM = 0.3108
SHOW_INLINE = True
OUTPUT_DIR = "figures"
FIX_TIME_AXIS = True
FIX_POS_AXIS = True
TIME_TICK_STEP_MS = 300
AXIS_MARGIN = 0.03
SCREEN_HALF_EXTENT_PX = 750

FIX_ACCEL_SCALE = True
ACCEL_SCALE_PERCENTILE = 99.0

DECISION_EPOCH = 6
MOVEMENT_EPOCH = 7
TARGET_HOLD_EPOCH = 8
MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)

_EPOCH_NAME_TO_CODE = {
    "REACTION": DECISION_EPOCH, "DECISION": DECISION_EPOCH,
    "DECISION_TIME": DECISION_EPOCH, "DECISIONTIME": DECISION_EPOCH,
    "MOVEMENT": MOVEMENT_EPOCH, "MOVE": MOVEMENT_EPOCH, "EXECUTION": MOVEMENT_EPOCH,
    "TARGET_HOLD": TARGET_HOLD_EPOCH, "TARGETHOLD": TARGET_HOLD_EPOCH, "HOLD": TARGET_HOLD_EPOCH,
}


def _epoch_to_code(v):
    """Numeric TaskEpoch code from a numeric OR string Epoch cell (NaN if unknown)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    try:
        return int(v)
    except (ValueError, TypeError):
        return _EPOCH_NAME_TO_CODE.get(str(v).strip().upper(), np.nan)

CENTER_TO_TARGET_DIST_PX = 320
TARGET_DIST_SCALE = 1.27
TARGET_DIAMETER_PX = 180
CENTER_DIAMETER_PX = 200
TARGET_DIST_PX = CENTER_TO_TARGET_DIST_PX * TARGET_DIST_SCALE
TARGET_RADIUS_PX = TARGET_DIAMETER_PX / 2.0
CENTER_RADIUS_PX = CENTER_DIAMETER_PX / 2.0
SCREEN_WIDTH_PX = 1920
SCREEN_HEIGHT_PX = 1080
USE_SCREEN_CENTER = True
SCREEN_VIEW_FULL = True

CM_PER_PX = PIXEL_PITCH_MM / 10.0

FONT_CANDIDATES = ["fonts/harding.ttf", "fonts/Harding.ttf"]
FONT_GLOBS = ["/content/*/fonts/harding.ttf", "/content/*/fonts/Harding.ttf"]


def _resolve_font():
    for cand in FONT_CANDIDATES:
        if Path(cand).exists():
            return cand
    for pattern in FONT_GLOBS:
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    return None


def _register_font(path):
    try:
        if path and Path(path).exists():
            fm.fontManager.addfont(path)
            plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
            return True
    except Exception:
        pass
    return False


plt.rcParams["axes.unicode_minus"] = False
FONT_PATH = _resolve_font()
if _register_font(FONT_PATH):
    print(f"Using font: {FONT_PATH}")
else:
    print("Harding font not found (looked in fonts/harding.ttf and /content/*/fonts/); using default font.")


FIGURE_DPI = 300
BASE_FONT_SIZE = 10

# Figure ink, storytelling-with-data style: everything that is context is grey,
# colour is spent only on the mark the reader should look at. Category colours
# stay the rig's (ColorCategoryMap.m), because a figure and the screen must agree.
INK = "#333333"          # text, emphasised lines
GRAY = "#8c8c8c"         # context marks, error bars, reference lines
GRAY_LIGHT = "#d9d9d9"   # de-emphasised fills, bands, chance lines
ACCENT = "#31688e"       # the one series the panel is about
ACCENT_2 = "#d44842"     # a second, warm accent (incorrect, a contrasting series)


def apply_figure_style(dpi=FIGURE_DPI, base=BASE_FONT_SIZE):
    """One font, one export resolution and one ink for every figure.

    rcParams are global to the kernel, so calling this once here -- before
    anything is drawn -- is what makes the whole notebook consistent instead
    of each cell setting its own sizes. savefig.dpi is set as well as the
    explicit dpi= arguments, so even a figure saved without one comes out at
    the same resolution. font.family is left alone: it was just set from
    harding.ttf above, and overriding it here would undo that.

    The chrome follows the storytelling-with-data rules: no top/right spines,
    grey hairline axes and ticks, no grid, frameless legends, left-aligned
    titles, so the data ink is the darkest thing on the page.
    """
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": dpi,
        "savefig.bbox": "tight",
        "font.size": base,
        "axes.titlesize": base + 1,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "figure.titlesize": base + 2,
        "axes.unicode_minus": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": GRAY,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.grid": False,
        "xtick.color": GRAY,
        "ytick.color": GRAY,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
        "lines.markersize": 5,
        "errorbar.capsize": 0,
    })


apply_figure_style()


class HampelScreen:
    MAD_SCALE = 1.4826

    def __init__(self, half_window=3, n_sigma=3.0):
        self.half_window = max(1, int(round(half_window)))
        self.n_sigma = float(n_sigma)

    def detect(self, x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        cleaned = x.copy()
        median_env = x.copy()
        threshold_env = np.zeros_like(x)
        mask = np.zeros_like(x, dtype=bool)
        if n < 3:
            return cleaned, mask, median_env, threshold_env
        h = self.half_window
        for col in range(d):
            source = x[:, col]
            for i in range(n):
                lo = max(0, i - h)
                hi = min(n, i + h + 1)
                window = source[lo:hi]
                med = np.median(window)
                mad = self.MAD_SCALE * np.median(np.abs(window - med))
                thr = self.n_sigma * mad
                median_env[i, col] = med
                threshold_env[i, col] = thr
                if mad > 0 and abs(source[i] - med) > thr:
                    cleaned[i, col] = med
                    mask[i, col] = True
        return cleaned, mask, median_env, threshold_env


class ButterworthLowpass:
    N_FACT = 6

    def __init__(self, cutoff_hz=20.0):
        self.cutoff_hz = float(cutoff_hz)

    def _design(self, fs):
        k = np.tan(np.pi * self.cutoff_hz / fs)
        norm = 1.0 / (1.0 + np.sqrt(2.0) * k + k ** 2)
        b = np.array([k ** 2, 2.0 * k ** 2, k ** 2]) * norm
        a = np.array([1.0, 2.0 * (k ** 2 - 1.0) * norm, (1.0 - np.sqrt(2.0) * k + k ** 2) * norm])
        return b, a

    @staticmethod
    def _steady_state(b, a):
        companion = np.array([[-a[1], 1.0], [-a[2], 0.0]])
        rhs = np.array([b[1] - a[1] * b[0], b[2] - a[2] * b[0]])
        return np.linalg.solve(np.eye(2) - companion, rhs)

    def __call__(self, x, fs):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        y = x.copy()
        if n == 0 or fs <= 0 or self.cutoff_hz <= 0 or self.cutoff_hz >= fs / 2.0:
            return y, False
        if n <= self.N_FACT:
            return y, False
        b, a = self._design(fs)
        zi = self._steady_state(b, a)
        nf = self.N_FACT
        for col in range(d):
            series = x[:, col]
            pre = 2.0 * series[0] - series[nf:0:-1]
            post = 2.0 * series[-1] - series[-2:-nf - 2:-1]
            ext = np.concatenate([pre, series, post])
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            y[:, col] = ext[nf:-nf]
        return y, True


class KalmanTrajectorySmoother:
    def __init__(self, meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.meas_sigma_px = float(meas_sigma_px)
        self.jerk_sigma_px_s3 = float(jerk_sigma_px_s3)
        self.gate_sigma = float(gate_sigma)

    def __call__(self, t, xy):
        t = np.asarray(t, dtype=float).ravel()
        xy = np.atleast_2d(np.asarray(xy, dtype=float))
        if xy.shape[0] == 1 and xy.shape[1] > 1:
            xy = xy.T
        n = t.size
        smoothed = xy.copy()
        n_gated = 0
        if n < 3 or xy.shape[0] != n:
            return smoothed, n_gated
        r = self.meas_sigma_px ** 2
        q = self.jerk_sigma_px_s3 ** 2
        dt_all = np.diff(t)
        acc_std0 = self.jerk_sigma_px_s3 * 20.0 * np.median(dt_all)
        gate2 = self.gate_sigma ** 2
        for col in range(xy.shape[1]):
            z = xy[:, col]
            xf = np.zeros((3, n))
            pf = np.zeros((3, 3, n))
            xp = np.zeros((3, n))
            pp = np.zeros((3, 3, n))
            fk = np.zeros((3, 3, n))
            dt0 = dt_all[0]
            xp[:, 0] = [z[0], (z[1] - z[0]) / dt0, 0.0]
            pp[:, :, 0] = np.diag([r, 2.0 * r / dt0 ** 2, acc_std0 ** 2])
            fk[:, :, 0] = np.eye(3)
            for k in range(n):
                if k > 0:
                    dt = dt_all[k - 1]
                    f = np.array([[1.0, dt, dt ** 2 / 2.0],
                                  [0.0, 1.0, dt],
                                  [0.0, 0.0, 1.0]])
                    qm = q * np.array([[dt ** 5 / 20.0, dt ** 4 / 8.0, dt ** 3 / 6.0],
                                       [dt ** 4 / 8.0, dt ** 3 / 3.0, dt ** 2 / 2.0],
                                       [dt ** 3 / 6.0, dt ** 2 / 2.0, dt]])
                    fk[:, :, k] = f
                    xp[:, k] = f @ xf[:, k - 1]
                    pp[:, :, k] = f @ pf[:, :, k - 1] @ f.T + qm
                innov = z[k] - xp[0, k]
                s = pp[0, 0, k] + r
                if innov ** 2 <= gate2 * s:
                    gain = pp[:, 0, k] / s
                    xf[:, k] = xp[:, k] + gain * innov
                    pf[:, :, k] = pp[:, :, k] - np.outer(gain, pp[0, :, k])
                    pf[:, :, k] = (pf[:, :, k] + pf[:, :, k].T) / 2.0
                else:
                    xf[:, k] = xp[:, k]
                    pf[:, :, k] = pp[:, :, k]
                    n_gated += 1
            xs = xf.copy()
            for k in range(n - 2, -1, -1):
                f = fk[:, :, k + 1]
                c = pf[:, :, k] @ f.T @ np.linalg.inv(pp[:, :, k + 1])
                xs[:, k] = xf[:, k] + c @ (xs[:, k + 1] - xp[:, k + 1])
            smoothed[:, col] = xs[0, :]
        return smoothed, n_gated


@dataclass
class KinematicSummary:
    peak_vel_cm: float = float("nan")
    mean_vel_cm: float = float("nan")
    median_vel_cm: float = float("nan")
    peak_accel_cm: float = float("nan")
    mean_accel_cm: float = float("nan")
    median_accel_cm: float = float("nan")
    vel_time_ms: np.ndarray = field(default_factory=lambda: np.array([]))
    vel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    accel_cm: np.ndarray = field(default_factory=lambda: np.array([]))
    move_takeoff_ms: float = float("nan")
    move_takeoff_alt_ms: float = float("nan")
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


@dataclass
class TrialResult:
    raw_time_ms: np.ndarray
    raw_xy: np.ndarray
    grid_time_ms: np.ndarray
    grid_xy: np.ndarray
    outlier_mask: np.ndarray
    hampel_median: np.ndarray
    hampel_threshold: np.ndarray
    n_replaced: int
    butter_applied: bool
    under_resolved: bool
    peak_vel_cm: float
    mean_vel_cm: float
    median_vel_cm: float
    peak_accel_cm: float
    mean_accel_cm: float
    median_accel_cm: float
    vel_time_ms: np.ndarray
    vel_cm: np.ndarray
    accel_cm: np.ndarray
    move_onset_ms: float = float("nan")
    move_offset_ms: float = float("nan")
    move_takeoff_ms: float = float("nan")
    n_duplicate_ts: int = 0
    move_takeoff_alt_ms: float = float("nan")
    peak_vel_time_ms: float = float("nan")
    window_start_ms: float = float("nan")
    window_end_ms: float = float("nan")


class TrajectoryProcessor:
    def __init__(self, grid_dt_s=0.008, cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX,
                min_move_samples=5, min_move_dur_s=None,
                hampel_half_window=3, hampel_n_sigma=3.0,
                meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.grid_dt_s = grid_dt_s
        self.cutoff_hz = cutoff_hz
        self.cm_per_px = cm_per_px
        self.min_move_samples = min_move_samples
        self.min_move_dur_s = min_move_dur_s
        self.hampel = HampelScreen(hampel_half_window, hampel_n_sigma)
        self.butter = ButterworthLowpass(cutoff_hz)
        self.kalman = KalmanTrajectorySmoother(meas_sigma_px, jerk_sigma_px_s3, gate_sigma)

    @classmethod
    def for_profile(cls, profile, **kwargs):
        return cls(hampel_half_window=profile.hampel_half_window,
                   hampel_n_sigma=profile.hampel_n_sigma,
                   meas_sigma_px=profile.meas_sigma_px,
                   jerk_sigma_px_s3=profile.jerk_sigma_px_s3,
                   gate_sigma=profile.gate_sigma, **kwargs)

    @staticmethod
    def _savgol_accel(vel_cm, dt_s, window, poly):
        n = int(vel_cm.size)
        if n == 0:
            return np.array([])
        if n == 1:
            return np.array([0.0])
        if n < 5:
            return np.abs(np.gradient(vel_cm, dt_s))
        w = min(int(window), n if n % 2 == 1 else n - 1)
        if w % 2 == 0:
            w -= 1
        w = max(w, 5)
        p = min(int(poly), w - 1)
        return np.abs(savgol_filter(vel_cm, w, p, deriv=1, delta=dt_s))

    def _build_grid(self, t):
        total = t[-1] - t[0]
        if total < self.grid_dt_s:
            return np.array([t[0], t[-1]])
        grid = np.arange(t[0], t[-1] + 1e-12, self.grid_dt_s)
        if t[-1] - grid[-1] > self.grid_dt_s / 4.0:
            grid = np.append(grid, t[-1])
        else:
            grid[-1] = t[-1]
        return grid

    @staticmethod
    def _takeoff_index(vel_cm, pk_idx, frac):
        thr = frac * float(vel_cm[pk_idx])
        lo_idx = pk_idx
        while lo_idx > 0 and vel_cm[lo_idx - 1] >= thr:
            lo_idx -= 1
        return lo_idx

    def _kinematics(self, grid_time_ms, grid_xy, under_resolved, window_end_ms=None):
        if under_resolved:
            return KinematicSummary()
        t = grid_time_ms / 1000.0
        gdt = np.diff(t)
        if gdt.size < 1 or np.any(gdt <= 0):
            return KinematicSummary()
        vel = np.hypot(np.diff(grid_xy[:, 0]), np.diff(grid_xy[:, 1])) / gdt
        tv_ms = (t[:-1] + gdt / 2.0) * 1000.0
        vel_cm = vel * self.cm_per_px
        accel_cm = self._savgol_accel(vel_cm, float(np.median(gdt)), SAVGOL_WINDOW, SAVGOL_POLYORDER)
        summary = KinematicSummary(vel_time_ms=tv_ms, vel_cm=vel_cm, accel_cm=accel_cm)
        if window_end_ms is not None and np.isfinite(window_end_ms):
            search = tv_ms <= window_end_ms
        else:
            search = np.ones(tv_ms.shape, dtype=bool)
        if not search.any():
            search = np.ones(tv_ms.shape, dtype=bool)
        search_idx = np.where(search)[0]
        pk_idx = int(search_idx[np.argmax(vel_cm[search_idx])])
        if float(vel[pk_idx]) < 1e-6:
            return summary
        lo_idx = self._takeoff_index(vel_cm, pk_idx, MOVE_SPEED_FRAC)
        alt_idx = self._takeoff_index(vel_cm, pk_idx, MOVE_SPEED_FRAC_ALT)
        summary.move_takeoff_ms = float(tv_ms[lo_idx])
        summary.move_takeoff_alt_ms = float(tv_ms[alt_idx])
        mask = search & (np.arange(tv_ms.size) >= lo_idx)
        win_time = tv_ms[mask]
        win_vel = vel_cm[mask]
        summary.window_start_ms = float(win_time[0])
        summary.window_end_ms = float(win_time[-1])
        k = int(np.argmax(win_vel))
        summary.peak_vel_cm = float(win_vel[k])
        summary.peak_vel_time_ms = float(win_time[k])
        summary.mean_vel_cm = float(np.mean(win_vel))
        summary.median_vel_cm = float(np.median(win_vel))
        win_acc = accel_cm[mask] if accel_cm.size == vel_cm.size else np.array([])
        win_acc = win_acc[np.isfinite(win_acc)]
        if win_acc.size:
            summary.peak_accel_cm = float(np.percentile(win_acc, ACCEL_PEAK_PERCENTILE))
            summary.mean_accel_cm = float(np.mean(win_acc))
            summary.median_accel_cm = float(np.median(win_acc))
        return summary

    def process(self, time_ms, x, y, move_window_ms=None, hold_start_ms=None):
        t = np.asarray(time_ms, dtype=float) / 1000.0
        order = np.argsort(t, kind="stable")
        t = t[order]
        xy = np.column_stack([np.asarray(x, dtype=float)[order],
                            np.asarray(y, dtype=float)[order]])
        n_raw = t.size
        t_unique, keep = np.unique(t, return_index=True)
        n_duplicate_ts = int(n_raw - t_unique.size)
        t = t_unique
        xy = xy[keep]
        n = t.size
        under_resolved = n < self.min_move_samples
        if self.min_move_dur_s is not None and n >= 2:
            under_resolved = under_resolved or (t[-1] - t[0]) < self.min_move_dur_s
        screened, outlier_mask, median_env, threshold_env = self.hampel.detect(xy)
        n_replaced = int(outlier_mask.sum())
        stage1, _ = self.kalman(t, screened)
        if n < 2:
            grid = t.copy()
            grid_xy = stage1.copy()
            applied = False
        else:
            grid = self._build_grid(t)
            grid_xy = np.column_stack([
                PchipInterpolator(t, stage1[:, 0])(grid),
                PchipInterpolator(t, stage1[:, 1])(grid),
            ])
            grid_fs = 1.0 / self.grid_dt_s
            grid_xy, applied = self.butter(grid_xy, grid_fs)
        grid_time_ms = (grid - t[0]) * 1000.0
        t0_ms = t[0] * 1000.0
        move_window_ms0 = None
        move_onset_ms = float("nan")
        if move_window_ms is not None:
            move_window_ms0 = (move_window_ms[0] - t0_ms, move_window_ms[1] - t0_ms)
            move_onset_ms = move_window_ms0[0]
        move_offset_ms = (hold_start_ms - t0_ms) if hold_start_ms is not None else float("nan")
        window_end_ms = move_window_ms0[1] if move_window_ms0 is not None else None
        kin = self._kinematics(grid_time_ms, grid_xy, under_resolved, window_end_ms)
        return TrialResult(
            raw_time_ms=(t - t[0]) * 1000.0,
            raw_xy=xy,
            grid_time_ms=grid_time_ms,
            grid_xy=grid_xy,
            outlier_mask=outlier_mask,
            hampel_median=median_env,
            hampel_threshold=threshold_env,
            n_replaced=n_replaced,
            butter_applied=applied,
            under_resolved=under_resolved,
            peak_vel_cm=kin.peak_vel_cm,
            mean_vel_cm=kin.mean_vel_cm,
            median_vel_cm=kin.median_vel_cm,
            peak_accel_cm=kin.peak_accel_cm,
            mean_accel_cm=kin.mean_accel_cm,
            median_accel_cm=kin.median_accel_cm,
            vel_time_ms=kin.vel_time_ms,
            vel_cm=kin.vel_cm,
            accel_cm=kin.accel_cm,
            move_onset_ms=move_onset_ms,
            move_offset_ms=move_offset_ms,
            move_takeoff_ms=kin.move_takeoff_ms,
            n_duplicate_ts=n_duplicate_ts,
            move_takeoff_alt_ms=kin.move_takeoff_alt_ms,
            peak_vel_time_ms=kin.peak_vel_time_ms,
            window_start_ms=kin.window_start_ms,
            window_end_ms=kin.window_end_ms,
        )


INPUT_SOURCE_OVERRIDES = {}
RZ2_MAX_MEDIAN_STEP_MS = 3.0


@dataclass
class PipelineProfile:
    hampel_half_window: int = 3
    hampel_n_sigma: float = 3.0
    meas_sigma_px: float = 2.0
    jerk_sigma_px_s3: float = 1e5
    gate_sigma: float = np.inf
    drop_unindexed_rows: bool = False


PIPELINE_PROFILES = {
    "rz2adc": PipelineProfile(drop_unindexed_rows=True),
    "usb": PipelineProfile(),
    "unknown": PipelineProfile(),
}


class InputSourceDetector:
    RZ2 = "rz2adc"
    USB = "usb"
    UNKNOWN = "unknown"
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]
    TAG_PREFIXES = ("trajectory_movement_", "trajectory_", "trial_data_", "trial_kinematics_")

    def __init__(self, overrides=None, max_rz2_step_ms=None):
        self.overrides = INPUT_SOURCE_OVERRIDES if overrides is None else overrides
        self.max_rz2_step_ms = RZ2_MAX_MEDIAN_STEP_MS if max_rz2_step_ms is None else max_rz2_step_ms

    @classmethod
    def run_tag(cls, path):
        name = Path(str(path)).stem
        for prefix in cls.TAG_PREFIXES:
            if name.startswith(prefix):
                return name[len(prefix):]
        return name

    @classmethod
    def median_step_ms(cls, frame):
        if "Time_ms" not in frame.columns:
            return float("nan")
        keys = [k for k in cls.GROUP_KEYS if k in frame.columns]
        d = frame.assign(_t=pd.to_numeric(frame["Time_ms"], errors="coerce")).dropna(subset=["_t"])
        if keys:
            d = d.sort_values(keys + ["_t"], kind="stable")
            steps = d.groupby(keys)["_t"].diff()
        else:
            steps = d["_t"].sort_values(kind="stable").diff()
        steps = steps[steps > 0]
        return float(steps.median()) if len(steps) else float("nan")

    def detect(self, frame, path=None):
        tag = self.run_tag(path) if path is not None else ""
        for key, source in self.overrides.items():
            if key and (key == tag or key in tag):
                return source, "manual override in INPUT_SOURCE_OVERRIDES"
        if "RZ2Idx" in frame.columns and pd.to_numeric(frame["RZ2Idx"], errors="coerce").notna().any():
            return self.RZ2, "RZ2Idx has indexed rows"
        step = self.median_step_ms(frame)
        if not np.isfinite(step):
            return self.UNKNOWN, "no RZ2Idx values and no usable Time_ms steps"
        source = self.RZ2 if step < self.max_rz2_step_ms else self.USB
        return source, f"no RZ2Idx values; median sample step {step:.2f} ms"


@dataclass
class Trial:
    date: str
    block: int
    trial: int
    attempt: int
    time_ms: np.ndarray
    move_time_ms: np.ndarray
    x: np.ndarray
    y: np.ndarray
    fs_hz: float
    move_window_ms: tuple = None
    hold_start_ms: float = None
    hold_window_ms: tuple = None
    hold_max_speed_cm: float = None
    hold_excursion_px: float = None

    @property
    def n_samples(self):
        return int(self.x.size)

    @property
    def label(self):
        return f"b{self.block:02d}_t{self.trial:02d}_a{self.attempt:02d}"


class TrajectoryDataset:
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, csv_path, move_epochs=None, detector=None):
        """
        move_epochs: None = no epoch filtering (use every row in csv_path
        as-is). Otherwise a single TaskEpoch code (e.g. MOVEMENT_EPOCH) or
        an iterable of codes (e.g. MOVE_EPOCHS = (DECISION_EPOCH,
        MOVEMENT_EPOCH, TARGET_HOLD_EPOCH)) -- a row is kept when its Epoch
        matches ANY of them. This mirrors CenterOutTask.m's kinematicsEpochs
        / the ismember(...) filter TrialKinematics.m and
        SaveMovementTrajectory.m now use on the MATLAB side, applied here
        explicitly instead of trusting the CSV to already be restricted the
        way we expect. Note this can only ever KEEP rows that exist in
        csv_path already -- if the session was recorded before
        kinematicsEpochs included a given epoch, that epoch's rows were
        never exported and this filter finds nothing to add for it (see the
        module-level comment above MOVE_EPOCHS).
        """
        self.path = Path(csv_path)
        self.frame = pd.read_csv(self.path)
        if "Epoch" in self.frame.columns and not pd.api.types.is_numeric_dtype(self.frame["Epoch"]):
            mapped = self.frame["Epoch"].map(_epoch_to_code)
            n_bad = int(mapped.isna().sum())
            if n_bad:
                unknown = sorted(set(self.frame.loc[mapped.isna(), "Epoch"].astype(str)))[:6]
                print(f"WARNING: {n_bad} rows have unrecognized Epoch names {unknown}; "
                    f"left unmatched -- add them to _EPOCH_NAME_TO_CODE if needed.")
            self.frame["Epoch"] = mapped
            print(f"Epoch column was text; normalized to numeric codes "
                f"{sorted(set(self.frame['Epoch'].dropna().astype(int)))}.")
        self.input_source, self.source_reason = (detector or InputSourceDetector()).detect(self.frame, self.path)
        self.profile = PIPELINE_PROFILES.get(self.input_source, PIPELINE_PROFILES["unknown"])
        self.n_unindexed_dropped = 0
        print(f"Input source of '{self.path.name}': {self.input_source} ({self.source_reason}).")
        if self.profile.drop_unindexed_rows and "RZ2Idx" in self.frame.columns:
            unindexed = pd.to_numeric(self.frame["RZ2Idx"], errors="coerce").isna()
            self.n_unindexed_dropped = int(unindexed.sum())
            self.frame = self.frame[~unindexed]
            print(f"  Dropped {self.n_unindexed_dropped} rows with NaN RZ2Idx (cached or un-indexed samples).")
        elif self.profile.drop_unindexed_rows:
            print("  No RZ2Idx column (export older than v8.21): cached rows cannot be identified and are kept.")
        if move_epochs is not None:
            if "Epoch" not in self.frame.columns:
                print(f"WARNING: '{self.path.name}' has no Epoch column -- "
                    f"cannot filter to move_epochs={move_epochs}; using every row as-is.")
            else:
                epochs = np.atleast_1d(move_epochs)
                before = len(self.frame)
                self.frame = self.frame[self.frame["Epoch"].isin(epochs)]
                print(f"Epoch filter {tuple(epochs)}: kept {len(self.frame)}/{before} rows of '{self.path.name}'.")

    @staticmethod
    def estimate_sampling_rate(time_ms):
        time_ms = np.asarray(time_ms, dtype=float)
        if time_ms.size < 2:
            return float("nan")
        steps = np.diff(np.sort(time_ms))
        steps = steps[steps > 0]
        if steps.size == 0:
            return float("nan")
        return 1000.0 / float(np.mean(steps))

    @staticmethod
    def _move_time_ms(group):
        if "MoveTime_ms" in group.columns:
            return group["MoveTime_ms"].to_numpy(dtype=float)
        tms = group["Time_ms"].to_numpy(dtype=float)
        return tms - float(tms.min()) if tms.size else tms

    def trials(self):
        for (block, trial, attempt), group in self.frame.groupby(self.GROUP_KEYS, sort=True):
            group = group.sort_values("Time_ms")
            move_window_ms = None
            hold_start_ms = None
            hold_window_ms = None
            hold_max_speed_cm = None
            hold_excursion_px = None
            if "Epoch" in group.columns:
                ep = group["Epoch"].to_numpy()
                tms = group["Time_ms"].to_numpy(dtype=float)
                xs_ = group["X_px"].to_numpy(dtype=float)
                ys_ = group["Y_px"].to_numpy(dtype=float)
                mv = tms[ep == MOVEMENT_EPOCH]
                if mv.size:
                    move_window_ms = (float(mv.min()), float(mv.max()))
                hmask = ep == TARGET_HOLD_EPOCH
                hold = tms[hmask]
                if hold.size:
                    hold_start_ms = float(hold.min())
                if hold.size >= 2:
                    hx, hy = xs_[hmask], ys_[hmask]
                    hold_window_ms = (float(hold.min()), float(hold.max()))
                    hdt = np.diff(hold) / 1000.0
                    good = hdt > 0
                    if good.any():
                        seg = np.hypot(np.diff(hx), np.diff(hy))[good] / hdt[good] * CM_PER_PX
                        hold_max_speed_cm = float(seg.max()) if seg.size else None
                    hold_excursion_px = float(np.max(np.hypot(hx - hx[0], hy - hy[0])))
            yield Trial(
                date=str(group["Date"].iloc[0]),
                block=int(block),
                trial=int(trial),
                attempt=int(attempt),
                time_ms=group["Time_ms"].to_numpy(dtype=float),
                move_time_ms=self._move_time_ms(group),
                x=group["X_px"].to_numpy(dtype=float),
                y=group["Y_px"].to_numpy(dtype=float),
                fs_hz=self.estimate_sampling_rate(group["Time_ms"].to_numpy()),
                move_window_ms=move_window_ms,
                hold_start_ms=hold_start_ms,
                hold_window_ms=hold_window_ms,
                hold_max_speed_cm=hold_max_speed_cm,
                hold_excursion_px=hold_excursion_px,
            )

    def global_limits(self, margin=0.03):
        t_max = 0.0
        p_min, p_max = np.inf, -np.inf
        for tr in self.trials():
            t_max = max(t_max, float(tr.move_time_ms.max()))
            p_min = min(p_min, float(tr.x.min()), float(tr.y.min()))
            p_max = max(p_max, float(tr.x.max()), float(tr.y.max()))
        pad = (p_max - p_min) * margin
        return (0.0, t_max), (p_min - pad, p_max + pad)

    def global_space_limits(self, margin=0.05):
        x_min, x_max = np.inf, -np.inf
        y_min, y_max = np.inf, -np.inf
        for tr in self.trials():
            x_min = min(x_min, float(tr.x.min()))
            x_max = max(x_max, float(tr.x.max()))
            y_min = min(y_min, float(tr.y.min()))
            y_max = max(y_max, float(tr.y.max()))
        px = (x_max - x_min) * margin
        py = (y_max - y_min) * margin
        return (x_min - px, x_max + px), (y_min - py, y_max + py)


def normalize_trial_schema(td):
    """Canonicalize the timing columns of a trial_data_*.csv across engine
    generations, so old and new sessions can be analysed by the same code.

    Four layouts exist in the wild. BOTH header spellings of the total are
    accepted on input, and the older ReactionTime_s header names a DIFFERENT
    interval in two of them -- which is the whole reason this function exists:

      gen A (v2_2 / pre-rename)   DecisionTime_s, ReactionTime_s, TotalTime_s
                                  ReactionTime_s = leave-center -> reach-target
                                  TotalTime_s    = decision + execution
      gen B (interim)             DecisionTime_s, ExecutionTime_s, TotalTime_s
      gen C (v8.19)               DecisionTime_s, ExecutionTime_s, ReactionTime_s
                                  ReactionTime_s = decision + execution (TOTAL)
      gen D (v8.20 onward)        DecisionTime_s, ExecutionTime_s, TotalTime_s
                                  TotalTime_s    = decision + execution (TOTAL)

    gen A is separated from the rest by whether ExecutionTime_s is present,
    which is the one signal that distinguishes them:
      present -> a ReactionTime_s column, if any, is the TOTAL   (gen C)
      absent  -> ReactionTime_s is the EXECUTION time            (gen A)

    Canonical output, always these three whatever came in:
      DecisionTime_s   target-onset  -> leave-center
      ExecutionTime_s  leave-center  -> reach-target
      TotalTime_s      DecisionTime_s + ExecutionTime_s

    ReactionTime_s is dropped after being folded into TotalTime_s: keeping
    both would put two identical columns into the correlation heatmap and
    the PCA, where an exact duplicate is not a second measurement.
    """
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


class TrialInfoTable:
    """Per-trial lookups keyed on (Block, TrialNumInBlock, Attempt).

    bar_size_deg / direction_correct / is_correct / error_type always come
    from trial_data_<runTag>.csv.

    move_samples (NumMovementSamples) moved OUT of trial_data_<runTag>.csv
    into its own companion file, trial_kinematics_<runTag>.csv, on the
    MATLAB side (same runTag, same Block+TrialNumInBlock+Attempt key --
    see CenterOutTask.m). To stay usable on BOTH older sessions (column
    still sitting in trial_data) and current ones (column only in
    trial_kinematics), this checks trial_data first and falls back to the
    companion kinematics file, auto-detected by swapping the
    'trial_data_' filename prefix for 'trial_kinematics_' unless
    kinematics_path is given explicitly. Never raises on a missing column
    or missing file -- move_samples just comes back None, and the caller
    (TrialFigure.build) already falls back to the trial's own raw sample
    count in that case.
    """
    KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, path, kinematics_path=None):
        self.frame = pd.read_csv(path) if path and Path(path).exists() else None
        if self.frame is not None:
            self.frame = normalize_trial_schema(self.frame)

        if kinematics_path is None and path:
            guess = Path(path).with_name(Path(path).name.replace("trial_data_", "trial_kinematics_", 1))
            kinematics_path = guess if guess.exists() else None
        self.kin_frame = pd.read_csv(kinematics_path) if kinematics_path and Path(kinematics_path).exists() else None

    @staticmethod
    def _match(frame, block, trial, attempt):
        row = frame[(frame.Block == block) & (frame.TrialNumInBlock == trial) & (frame.Attempt == attempt)]
        return row.iloc[0] if len(row) == 1 else None

    def lookup(self, block, trial, attempt):
        result = {"bar_size_deg": None, "move_samples": None,
                  "direction_correct": None, "is_correct": None, "error_type": None,
                  "decision_time_s": None, "execution_time_s": None, "total_time_s": None}

        if self.frame is not None:
            row = self._match(self.frame, block, trial, attempt)
            if row is not None:
                result["bar_size_deg"] = float(row["BarSizeVA_deg"])
                result["direction_correct"] = str(row["DirectionCorrect"])
                result["is_correct"] = int(row["IsCorrect"])
                result["error_type"] = int(row["ErrorType"])
                if "DecisionTime_s" in row.index:
                    result["decision_time_s"] = float(row["DecisionTime_s"])
                if "ExecutionTime_s" in row.index:
                    result["execution_time_s"] = float(row["ExecutionTime_s"])
                if "TotalTime_s" in row.index:
                    result["total_time_s"] = float(row["TotalTime_s"])
                if "NumMovementSamples" in row.index:
                    result["move_samples"] = int(row["NumMovementSamples"])

        if result["move_samples"] is None and self.kin_frame is not None:
            krow = self._match(self.kin_frame, block, trial, attempt)
            if krow is not None and "NumMovementSamples" in krow.index:
                result["move_samples"] = int(krow["NumMovementSamples"])

        return result


class OfflineClockFit:
    SEED_RATE_HZ = 24414.0625 / 26.0

    def __init__(self, index, time_ms, source=""):
        idx = np.asarray(index, dtype=float)
        tms = np.asarray(time_ms, dtype=float)
        self.source = source
        self.n_rows = int(idx.size)
        self.n_indexed = int(np.isfinite(idx).sum())
        ok = np.isfinite(idx) & np.isfinite(tms) & (tms > 0)
        order = np.argsort(idx[ok], kind="stable")
        self.index = idx[ok][order]
        self.time_ms = tms[ok][order]
        self.slope_ms = float("nan")
        self.intercept_ms = float("nan")
        if self.index.size >= 3 and np.ptp(self.index) > 0:
            x = self.index - self.index.mean()
            y = self.time_ms - self.time_ms.mean()
            self.slope_ms = float(np.dot(x, y) / np.dot(x, x))
            self.intercept_ms = float(self.time_ms.mean() - self.slope_ms * self.index.mean())

    @classmethod
    def from_csv(cls, path):
        path = Path(path)
        frame = pd.read_csv(path)
        if "RZ2Idx" not in frame.columns or "Time_ms" not in frame.columns:
            return cls([], [], source=path.name)
        return cls(frame["RZ2Idx"].to_numpy(), frame["Time_ms"].to_numpy(), source=path.name)

    @property
    def fitted(self):
        return bool(np.isfinite(self.slope_ms) and self.slope_ms > 0)

    @property
    def rate_hz(self):
        return 1000.0 / self.slope_ms if self.fitted else float("nan")

    def predict_ms(self, index):
        return self.intercept_ms + self.slope_ms * np.asarray(index, dtype=float)

    def residual_ms(self):
        return self.time_ms - self.predict_ms(self.index)

    def summary(self):
        nan = float("nan")
        out = {"rows": self.n_rows, "indexed_rows": self.n_indexed,
               "unindexed_rows": self.n_rows - self.n_indexed,
               "rate_hz": self.rate_hz,
               "rate_dev_ppm": (self.rate_hz / self.SEED_RATE_HZ - 1.0) * 1e6 if self.fitted else nan,
               "resid_rms_ms": nan, "resid_p99_abs_ms": nan, "resid_max_abs_ms": nan}
        if self.fitted:
            r = self.residual_ms()
            out["resid_rms_ms"] = float(np.sqrt(np.mean(r ** 2)))
            out["resid_p99_abs_ms"] = float(np.percentile(np.abs(r), 99))
            out["resid_max_abs_ms"] = float(np.max(np.abs(r)))
        steps = np.diff(self.time_ms)
        back = steps[steps < 0]
        out["backward_steps"] = int(back.size)
        out["worst_backward_ms"] = float(-back.min()) if back.size else 0.0
        return out

    def report(self):
        if not self.fitted:
            return (f"Offline clock fit: no usable RZ2Idx rows in '{self.source}' "
                    f"(USB or mouse input, or an export older than v8.21); "
                    f"Time_ms is the only time base.")
        s = self.summary()
        return (f"Offline clock fit on '{self.source}' (OLS of Time_ms on RZ2Idx, "
                f"{s['indexed_rows']}/{s['rows']} rows indexed):\n"
                f"  rate {s['rate_hz']:.4f} Hz ({s['rate_dev_ppm']:+.0f} ppm vs seed "
                f"{self.SEED_RATE_HZ:.4f} Hz)\n"
                f"  online Time_ms minus offline fit: RMS {s['resid_rms_ms']:.2f} ms, "
                f"p99 |r| {s['resid_p99_abs_ms']:.2f} ms, max |r| {s['resid_max_abs_ms']:.2f} ms\n"
                f"  backward steps of Time_ms in index order: {s['backward_steps']} "
                f"(worst {s['worst_backward_ms']:.2f} ms)")


def estimate_screen_layout(dataset, info_table, target_dist=TARGET_DIST_PX,
                           target_radius=TARGET_RADIUS_PX, center_radius=CENTER_RADIUS_PX,
                           center_override=None):
    if center_override is not None:
        cx, cy = center_override
    else:
        directions = ["Right_0", "Up_90", "Left_180", "Down_270"]
        ends = {d: [] for d in directions}
        for trial in dataset.trials():
            info = info_table.lookup(trial.block, trial.trial, trial.attempt)
            d = info.get("direction_correct")
            if info.get("is_correct") == 1 and d in ends:
                ends[d].append([trial.x[-1], trial.y[-1]])
        means = {}
        for d, pts in ends.items():
            if pts:
                arr = np.asarray(pts)
                means[d] = (float(arr[:, 0].mean()), float(arr[:, 1].mean()))
        cx = cy = None
        if "Right_0" in means and "Left_180" in means:
            cx = (means["Right_0"][0] + means["Left_180"][0]) / 2.0
        if "Up_90" in means and "Down_270" in means:
            cy = (means["Up_90"][1] + means["Down_270"][1]) / 2.0
        if cx is None:
            xs = [p[0] for p in means.values()]
            cx = float(np.mean(xs)) if xs else None
        if cy is None:
            ys = [p[1] for p in means.values()]
            cy = float(np.mean(ys)) if ys else None
        if cx is None or cy is None:
            return None
    offsets = {"Right_0": (target_dist, 0.0), "Up_90": (0.0, -target_dist),
               "Left_180": (-target_dist, 0.0), "Down_270": (0.0, target_dist)}
    targets = {d: (cx + ox, cy + oy) for d, (ox, oy) in offsets.items()}
    return {"center": (cx, cy), "targets": targets,
            "target_radius": target_radius, "center_radius": center_radius}




In [ ]:
# --- feature engineering (single-session notebook, 1.2): per-trial kinematic
# features, palettes, pairplot, violins, correlation heatmap and the
# correct-vs-incorrect and accuracy tables. Definitions only. ---------------
try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"], check=True)
    import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EARLY_EXIT_COL = "ErrorType"
EARLY_EXIT_CODE = 1
EARLY_EXIT_CHECK_COL = "ChosenTarget"


def drop_early_exits(df, col=EARLY_EXIT_COL, code=EARLY_EXIT_CODE,
                     check_col=EARLY_EXIT_CHECK_COL, verbose=True):
    if col not in df.columns:
        print(f"WARNING: '{col}' not in the table; keeping all {len(df)} trials.")
        return df
    error_type = pd.to_numeric(df[col], errors="coerce")
    early = error_type == code
    out = df[~early].copy()
    if verbose:
        n_drop = int(early.sum())
        pct = 100.0 * n_drop / len(df) if len(df) else 0.0
        print(f"Early exits excluded: dropped {n_drop}/{len(df)} trials "
              f"({pct:.1f}%) with {col} == {code}; {len(out)} trials go into the figures.")
        n_unknown = int(error_type.isna().sum())
        if n_unknown:
            print(f"  WARNING: {n_unknown} trials have no numeric {col} and were kept.")
        if check_col in df.columns:
            no_choice = ~(pd.to_numeric(df[check_col], errors="coerce") >= 1)
            early_with_choice = int((early & ~no_choice).sum())
            kept_without_choice = int((~early & no_choice).sum())
            if early_with_choice or kept_without_choice:
                print(f"  WARNING: {col} and {check_col} disagree on "
                      f"{early_with_choice + kept_without_choice} trials: "
                      f"{early_with_choice} dropped trials have {check_col} >= 1 and "
                      f"{kept_without_choice} kept trials have no {check_col} >= 1.")
            else:
                print(f"  Check: {col} == {code} coincides with {check_col} < 1 on every trial.")
    return out


PAIRPLOT_FEATURES = ["BarSizeVA_deg", "DecisionTime_s", "ExecutionTime_s",
                     "n_samples", "fs_hz", "move_takeoff_ms",
                     "peak_vel_cm", "mean_vel_cm", "peak_accel_cm",
                     "path_length_cm", "straightness"]
VIOLIN_GROUP = "ChosenTarget"
CORRECTNESS_GROUP = "IsCorrect"
VIOLIN_FEATURES = ["DecisionTime_s", "ExecutionTime_s", "n_samples", "move_takeoff_ms",
                   "peak_vel_cm", "mean_vel_cm", "peak_accel_cm",
                   "path_length_cm", "straightness"]
HUE = "StimulusGroup"
CORRELATION_METHOD = "spearman"

CATEGORY_COLORS = {"ShortGroup": "#FFA500", "MidGroup": "#00FF00", "LongGroup": "#0000FF"}
CATEGORY_COLORS_BY_ID = {1: "#FFA500", 2: "#00FF00", 3: "#0000FF"}
CORRECTNESS_COLORS = {1: "#8c8c8c", 0: "#d44842"}   # correct = grey context, incorrect = the accent
# Trial dots on the violin panels. Mid grey, not near-white: the violins
# are drawn hollow, so the dots now sit against the white page instead of
# against a filled body, and #DDDDDD disappeared there.
POINT_COLOR = "#B3B3B3"
POINT_EDGE = "#555555"
CORRECTNESS_LABELS = {0: "incorrect", 1: "correct"}


def category_palette(levels):
    """Colours that match what the subject actually saw on screen.

    ColorCategoryMap.m paints category 1 ORANGE, 2 GREEN, 3 BLUE, and
    ConfigOrgParams.m exposes the same three as color3CatShort/Mid/Long
    (FFA500 / 00FF00 / 0000FF). Using anything else here means a figure and
    the rig disagree about which category is which. Levels that are not one
    of the three fall back to grey rather than silently borrowing a
    category's colour.
    """
    out = {}
    for lv in levels:
        key = lv
        if isinstance(lv, (int, float)) and not isinstance(lv, bool):
            key = int(lv)
        out[lv] = CATEGORY_COLORS.get(key, CATEGORY_COLORS_BY_ID.get(key, "#999999"))
    return out


def correctness_palette(levels):
    """Grey for correct, one warm accent for incorrect.

    Correct trials are the majority and the context; the incorrect ones are
    what the reader is looking for, so they get the only colour. Deliberately
    outside the category palette, so a correct/incorrect split can never be
    mistaken for a category split."""
    return {lv: CORRECTNESS_COLORS.get(int(lv), "#999999") for lv in levels}
ID_COLS = ["Block", "TrialNumInBlock", "Attempt"]


def _path_metrics(result, cm_per_px):
    gt = result.grid_time_ms
    gxy = result.grid_xy
    if gt.size < 2:
        return np.nan, np.nan
    lo = result.move_onset_ms if np.isfinite(result.move_onset_ms) else gt[0]
    hi = result.move_offset_ms if np.isfinite(result.move_offset_ms) else gt[-1]
    mask = (gt >= lo) & (gt <= hi)
    if mask.sum() < 2:
        mask = np.ones(gt.shape, dtype=bool)
    seg = gxy[mask]
    steps = np.hypot(np.diff(seg[:, 0]), np.diff(seg[:, 1]))
    path_len = float(steps.sum()) * cm_per_px
    net = float(np.hypot(seg[-1, 0] - seg[0, 0], seg[-1, 1] - seg[0, 1])) * cm_per_px
    straight = net / path_len if path_len > 1e-9 else np.nan
    return path_len, straight


def build_trial_features(traj_path, trial_data_path, move_epochs=None):
    dataset = TrajectoryDataset(traj_path, move_epochs=move_epochs)
    processor = TrajectoryProcessor.for_profile(dataset.profile, grid_dt_s=GRID_DT_S,
                                                cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX)
    recs = []
    n_dup_rows = 0
    n_dup_trials = 0
    for tr in dataset.trials():
        r = processor.process(tr.time_ms, tr.x, tr.y, tr.move_window_ms, tr.hold_start_ms)
        n_dup_rows += r.n_duplicate_ts
        n_dup_trials += int(r.n_duplicate_ts > 0)
        path_len, straight = _path_metrics(r, CM_PER_PX)
        hold_dur_ms = (tr.hold_window_ms[1] - tr.hold_window_ms[0]) if tr.hold_window_ms else np.nan
        recs.append({
            "Block": tr.block, "TrialNumInBlock": tr.trial, "Attempt": tr.attempt,
            "fs_hz": tr.fs_hz, "n_samples": tr.n_samples,
            "peak_vel_cm": r.peak_vel_cm, "mean_vel_cm": r.mean_vel_cm, "median_vel_cm": r.median_vel_cm,
            "peak_accel_cm": r.peak_accel_cm, "mean_accel_cm": r.mean_accel_cm, "median_accel_cm": r.median_accel_cm,
            "path_length_cm": path_len, "straightness": straight,
            "move_takeoff_ms": r.move_takeoff_ms,
            "takeoff_lead_ms": r.move_onset_ms - r.move_takeoff_ms,
            "hold_dur_ms": hold_dur_ms, "hold_max_speed_cm": tr.hold_max_speed_cm,
            "hold_excursion_px": tr.hold_excursion_px,
            "under_resolved": int(r.under_resolved),
        })
    kin = pd.DataFrame.from_records(recs)
    td = normalize_trial_schema(pd.read_csv(trial_data_path))
    df = td.merge(kin, on=ID_COLS, how="left")
    if "DecisionTime_s" in df.columns:
        df["TakeoffTime_s"] = (pd.to_numeric(df["DecisionTime_s"], errors="coerce")
                               - pd.to_numeric(df["takeoff_lead_ms"], errors="coerce") / 1000.0)
    df["InputSource"] = dataset.input_source
    df.attrs["input_source"] = dataset.input_source
    df.attrs["source_reason"] = dataset.source_reason
    df.attrs["n_unindexed_dropped"] = dataset.n_unindexed_dropped
    df.attrs["n_duplicate_ts"] = n_dup_rows
    df.attrs["n_trials_with_duplicates"] = n_dup_trials
    return df


def pairplot_trials(df, features=PAIRPLOT_FEATURES, hue=HUE):
    cols = [c for c in features if c in df.columns]
    missing = [c for c in features if c not in df.columns]
    if missing:
        print(f"WARNING: requested features not in the table, skipped: {missing}. "
              f"Available columns: {list(df.columns)}")
    use_hue = hue if hue in df.columns else None
    sub = df.dropna(subset=cols).copy()
    print(f"Pairplot on {len(sub)}/{len(df)} trials (rows with any NaN in {cols} dropped).")
    pal = None
    if use_hue == "StimulusGroup":
        pal = category_palette(sorted(pd.unique(sub[use_hue])))
    elif use_hue == CORRECTNESS_GROUP:
        pal = correctness_palette(sorted(pd.unique(sub[use_hue])))
    g = sns.pairplot(sub, vars=cols, hue=use_hue, palette=pal, corner=True,
                     diag_kind="kde", plot_kws=dict(s=20, alpha=0.6, edgecolor="none"))
    g.figure.suptitle("Per-trial feature pairplot", y=1.02)
    g.figure.savefig("trial_pairplot.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return g


def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    ok = np.isfinite(p)
    out = np.full(p.shape, np.nan)
    pv = p[ok]
    n = pv.size
    if n:
        order = np.argsort(pv)
        ranked = pv[order] * n / (np.arange(n) + 1)
        q_sorted = np.minimum.accumulate(ranked[::-1])[::-1]
        q = np.empty(n)
        q[order] = np.clip(q_sorted, 0, 1)
        out[ok] = q
    return out


def violin_by_group(df, features=VIOLIN_FEATURES, group_col=VIOLIN_GROUP, alpha=0.05,
                    palette=None, savename=None):
    from scipy import stats
    if group_col not in df.columns:
        print(f"'{group_col}' not in the table; skipping violin plots.")
        return None
    feats = [f for f in features if f in df.columns]
    d = df[[group_col] + feats].copy()
    d = d[d[group_col].notna()]
    levels = sorted(pd.unique(d[group_col]))
    if len(levels) < 2:
        print(f"'{group_col}' has < 2 levels ({levels}); nothing to compare.")
        return None
    ncol = 3
    nrow = int(np.ceil(len(feats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    rows = []
    for i, f in enumerate(feats):
        ax = axes[i]
        sub = d[[group_col, f]].dropna()
        arrays = [sub.loc[sub[group_col] == lv, f].to_numpy() for lv in levels]
        arrays = [a for a in arrays if a.size > 0]
        H = p = np.nan
        if len(arrays) >= 2 and all(a.size >= 1 for a in arrays) and sub[f].nunique() > 1:
            H, p = stats.kruskal(*arrays)
        pal = palette
        if pal is None:
            pal = (category_palette(levels) if group_col in ("StimulusGroup", "ChosenTarget")
                   else correctness_palette(levels) if group_col == CORRECTNESS_GROUP else None)
        # Hollow violins with the boxplot inside. The outline carries the
        # category colour and the box (IQR bar, whisker line, white median
        # dot) sits in the middle, so the SHAPE of the distribution and its
        # median/IQR are both readable without one hiding the other -- which
        # the filled violin with quartile lines could not do. Points go down
        # FIRST so they stay under the outline and the box, instead of
        # peppering over the summary that is the point of the panel.
        sns.stripplot(x=group_col, y=f, data=sub, order=levels, ax=ax,
                      color=POINT_COLOR, size=2.4, jitter=0.28, alpha=0.55,
                      edgecolor=POINT_EDGE, linewidth=0.2, zorder=0.5)
        sns.violinplot(x=group_col, y=f, data=sub, order=levels, ax=ax, hue=group_col,
                       palette=pal, legend=False, inner="box", cut=0,
                       density_norm="width", fill=False, linewidth=1.6,
                       inner_kws=dict(box_width=5.5, whis_width=1.4))
        ax.set_title(f"{f}  (KW p={p:.3f})" if np.isfinite(p) else f"{f}  (KW n/a)",
                     fontsize=9)
        ax.set_xlabel(group_col, fontsize=8)
        ax.set_ylabel(f, fontsize=8)
        ax.tick_params(labelsize=7)
        rows.append({"feature": f, "kruskal_H": H, "kruskal_p": p})
    for j in range(len(feats), len(axes)):
        axes[j].axis("off")
    fig.suptitle(f"Per-trial features by {group_col} (Kruskal-Wallis per feature)", y=1.0)
    fig.tight_layout()
    fig.savefig(savename or f"violin_by_{group_col.lower()}.png", dpi=FIGURE_DPI, bbox_inches="tight")
    res = pd.DataFrame(rows)
    res["q_fdr"] = _bh_fdr(res["kruskal_p"].to_numpy())
    res[f"sig(FDR<{alpha})"] = np.where(res["q_fdr"] < alpha, "yes", "no")
    res = res.reindex(res["kruskal_p"].fillna(1).sort_values().index).reset_index(drop=True)
    print(f"\nDoes each feature differ across {group_col} levels {levels}? "
          f"(Kruskal-Wallis, FDR-corrected):\n")
    print(res.round(4).to_string(index=False))
    n_sig = int((res["q_fdr"] < alpha).sum())
    print(f"\n{n_sig}/{len(res)} features differ significantly across {group_col} at FDR < {alpha}.")
    return res


def correlation_heatmap(df, method=CORRELATION_METHOD):
    num = df.select_dtypes("number").drop(columns=[c for c in ID_COLS if c in df.columns],
                                           errors="ignore")
    num = num.loc[:, num.nunique(dropna=True) > 1]
    corr = num.corr(method=method)
    fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 3, 0.6 * len(corr) + 2))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
                square=True, cbar_kws={"shrink": 0.8}, ax=ax,
                annot_kws={"size": 7})
    ax.set_title(f"{method.capitalize()} correlation of per-trial features")
    fig.tight_layout()
    fig.savefig("trial_corr_heatmap.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return corr


def correct_vs_incorrect(df, features=VIOLIN_FEATURES, alpha=0.05,
                         group_col=CORRECTNESS_GROUP):
    """Per-feature comparison of correct against incorrect trials.

    The violins above split by ChosenTarget, which asks whether the movement
    differs by WHICH category the subject picked. This asks the other
    question: whether it differs by whether the pick was RIGHT. The two come
    apart, because a subject can reach the wrong target with a perfectly
    ordinary movement, and the interesting case is when they cannot.

    Mann-Whitney per feature with a rank-biserial effect size, FDR-corrected
    across features. Restricted to trials that reached a target, so an early
    exit (no choice at all) is not scored as an incorrect categorisation.
    """
    from scipy import stats
    if group_col not in df.columns:
        print(f"'{group_col}' not in the table; correct/incorrect comparison skipped.")
        return None
    d = df.copy()
    if "ChosenTarget" in d.columns:
        d = d[pd.to_numeric(d["ChosenTarget"], errors="coerce") >= 1]
    d[group_col] = pd.to_numeric(d[group_col], errors="coerce")
    d = d[d[group_col].isin([0, 1])]
    n_ok, n_bad = int((d[group_col] == 1).sum()), int((d[group_col] == 0).sum())
    print(f"Correct vs incorrect on {len(d)} trials that reached a target "
          f"({n_ok} correct, {n_bad} incorrect).")
    if n_ok < 5 or n_bad < 5:
        print("  Too few trials on one side for a meaningful comparison.")
        return None
    rows = []
    for f in [c for c in features if c in d.columns]:
        s = d[[group_col, f]].apply(pd.to_numeric, errors="coerce").dropna()
        a = s.loc[s[group_col] == 1, f].to_numpy()
        b = s.loc[s[group_col] == 0, f].to_numpy()
        if a.size < 5 or b.size < 5 or s[f].nunique() < 2:
            rows.append({"feature": f, "median_correct": np.nan, "median_incorrect": np.nan,
                         "U": np.nan, "p": np.nan, "rank_biserial": np.nan})
            continue
        U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append({"feature": f,
                     "median_correct": float(np.median(a)),
                     "median_incorrect": float(np.median(b)),
                     "delta": float(np.median(a) - np.median(b)),
                     "n_correct": a.size, "n_incorrect": b.size,
                     "U": float(U), "p": float(p),
                     "rank_biserial": float(2.0 * U / (a.size * b.size) - 1.0)})
    res = pd.DataFrame(rows)
    res["q_fdr"] = _bh_fdr(res["p"].to_numpy())
    res[f"sig(FDR<{alpha})"] = np.where(res["q_fdr"] < alpha, "yes", "no")
    res = res.reindex(res["p"].fillna(1).sort_values().index).reset_index(drop=True)
    print(f"\nDoes each feature differ between correct and incorrect trials? "
          f"(Mann-Whitney, FDR-corrected):\n")
    print(res.round(4).to_string(index=False))
    print(f"\n{int((res['q_fdr'] < alpha).sum())}/{len(res)} features differ at FDR < {alpha}. "
          f"rank_biserial > 0 means the feature is LARGER on correct trials.")
    return res


def accuracy_breakdown(df):
    """Accuracy by true category and by target position, in one table.

    Category accuracy is what the psychometric section models; position
    accuracy is the motor control, and it should be FLAT because the correct
    position is randomised independently of bar length. A position that
    stands out is a rig or a handedness effect, not a categorisation result.
    """
    out = {}
    if "StimulusGroup" in df.columns:
        g = df.groupby("StimulusGroup")["IsCorrect"].agg(["mean", "count"])
        g.columns = ["accuracy", "n"]
        out["by_category"] = g
        print("\nAccuracy by true category:")
        print(g.round(4).to_string())
    if "DirectionCorrect" in df.columns:
        g = df.groupby("DirectionCorrect")["IsCorrect"].agg(["mean", "count"])
        g.columns = ["accuracy", "n"]
        out["by_direction"] = g
        print("\nAccuracy by correct target position:")
        print(g.round(4).to_string())
    fig, axes = plt.subplots(1, len(out), figsize=(6.0 * len(out), 4.0), squeeze=False)
    for ax, (name, g) in zip(axes.ravel(), out.items()):
        cols = ([CATEGORY_COLORS.get(i, "#999999") for i in g.index]
                if name == "by_category" else "#31688e")
        se = np.sqrt(g["accuracy"] * (1 - g["accuracy"]) / g["n"])
        ax.bar(g.index.astype(str), g["accuracy"], yerr=1.96 * se, color=cols,
               edgecolor="none", alpha=0.9, width=0.6, ecolor=GRAY, capsize=0)
        ax.axhline(1.0 / 3.0, color=GRAY_LIGHT, lw=1)
        ax.text(len(g) - 0.5, 1.0 / 3.0 + 0.015, "chance", ha="right", va="bottom",
                fontsize=8, color=GRAY)
        ax.set_ylim(0, 1.02)
        ax.set_ylabel("P(correct)")
        ax.set_title(name.replace("_", " "))
        ax.tick_params(axis="x", rotation=20, labelsize=8)
    fig.suptitle("Accuracy by true category and by target position", y=1.02)
    fig.tight_layout()
    fig.savefig("accuracy_breakdown.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return out


In [ ]:
from scipy import stats

# --- psychometric fit (single-session notebook, 1.5) ------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

CATEGORY_COLORS = {"ShortGroup": "#FFA500", "MidGroup": "#00FF00", "LongGroup": "#0000FF"}

CATEGORY_ORDER = ["ShortGroup", "MidGroup", "LongGroup"]
CATEGORY_ID = {name: i + 1 for i, name in enumerate(CATEGORY_ORDER)}
STIMULUS_COL = "BarSizeVA_deg"
CHOICE_COL = "ChosenTarget"
TRUTH_COL = "StimulusGroup"
MIN_LEVELS_FOR_FIT = 4
N_BOOTSTRAP = 400
LAPSE_MAX = 0.35
PSYCHO_SEED = 0


def ordinal_choice_table(df, stimulus_col=STIMULUS_COL, choice_col=CHOICE_COL):
    """One row per (stimulus level, ordinal split) with binomial counts.

    A K-category ordinal task has K-1 boundaries. Split j (j = 2..K) asks
    'did the subject choose category j or higher?', so each boundary gets its
    own two-alternative psychometric curve fitted on the same trials. Only
    trials with a real choice (ChosenTarget >= 1) contribute: an early exit
    is a missing response, not a response at the bottom category.
    """
    d = df[df[choice_col].notna()].copy()
    d = d[pd.to_numeric(d[choice_col], errors="coerce") >= 1]
    if d.empty:
        return pd.DataFrame(columns=["split", "x", "n", "k", "p"])
    d["_choice"] = pd.to_numeric(d[choice_col], errors="coerce").astype(int)
    kmax = int(d["_choice"].max())
    rows = []
    for j in range(2, kmax + 1):
        for x, g in d.groupby(stimulus_col):
            n = len(g)
            k = int((g["_choice"] >= j).sum())
            rows.append({"split": j, "x": float(x), "n": n, "k": k, "p": k / n if n else np.nan})
    return pd.DataFrame(rows).sort_values(["split", "x"]).reset_index(drop=True)


def _neg_log_lik(theta, x, n, k):
    mu, log_sigma, logit_lapse = theta
    sigma = np.exp(np.clip(log_sigma, -20, 20))
    lapse = LAPSE_MAX / (1.0 + np.exp(-np.clip(logit_lapse, -30, 30)))
    p = lapse / 2.0 + (1.0 - lapse) * norm.cdf((x - mu) / sigma)
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.sum(k * np.log(p) + (n - k) * np.log(1.0 - p))


def fit_psychometric(x, n, k):
    """Cumulative-Gaussian fit with a lapse rate, by binomial maximum likelihood.

    p(x) = lapse/2 + (1 - lapse) * Phi((x - mu) / sigma)

    mu is the boundary (the 50% point once lapses are accounted for) and sigma
    is inversely proportional to sensitivity. The lapse term matters here
    because motor errors (ErrorType 1 and 3) put a floor and a ceiling on the
    curve; without it those trials flatten the slope and bias mu.
    """
    x, n, k = np.asarray(x, float), np.asarray(n, float), np.asarray(k, float)
    if x.size < 3 or np.unique(x).size < 3:
        return None
    p_obs = k / np.maximum(n, 1)
    mu0 = float(np.interp(0.5, p_obs, x)) if np.all(np.diff(p_obs) >= 0) else float(np.mean(x))
    span = float(x.max() - x.min()) or 1.0
    best = None
    for s0 in (span / 8.0, span / 4.0, span / 2.0):
        theta0 = np.array([mu0, np.log(s0), -2.0])
        try:
            r = minimize(_neg_log_lik, theta0, args=(x, n, k), method="Nelder-Mead",
                         options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-6})
        except Exception:
            continue
        if r.success or r.fun < np.inf:
            if best is None or r.fun < best.fun:
                best = r
    if best is None:
        return None
    mu, log_sigma, logit_lapse = best.x
    return {"mu": float(mu), "sigma": float(np.exp(np.clip(log_sigma, -20, 20))),
            "lapse": float(LAPSE_MAX / (1.0 + np.exp(-np.clip(logit_lapse, -30, 30)))),
            "neg_log_lik": float(best.fun), "n_trials": int(n.sum())}


def bootstrap_psychometric(x, n, k, n_boot=N_BOOTSTRAP, seed=PSYCHO_SEED):
    """Percentile CIs by resampling each stimulus level's binomial counts."""
    rng = np.random.default_rng(seed)
    x, n, k = np.asarray(x, float), np.asarray(n, int), np.asarray(k, int)
    mus, sigmas = [], []
    p_hat = k / np.maximum(n, 1)
    for _ in range(n_boot):
        kb = rng.binomial(n, np.clip(p_hat, 0, 1))
        f = fit_psychometric(x, n, kb)
        if f is not None and np.isfinite(f["mu"]):
            mus.append(f["mu"]); sigmas.append(f["sigma"])
    if not mus:
        return {}
    return {"mu_lo": float(np.percentile(mus, 2.5)), "mu_hi": float(np.percentile(mus, 97.5)),
            "sigma_lo": float(np.percentile(sigmas, 2.5)), "sigma_hi": float(np.percentile(sigmas, 97.5)),
            "n_boot_ok": len(mus)}


def fit_all_boundaries(df, n_boot=N_BOOTSTRAP, verbose=True):
    """Fit one curve per ordinal split; returns (fits DataFrame, counts table)."""
    tab = ordinal_choice_table(df)
    if tab.empty:
        print("No trials with a real choice; psychometric fit skipped.")
        return pd.DataFrame(), tab
    out = []
    for j, g in tab.groupby("split"):
        levels = g["x"].nunique()
        rec = {"split": j, "boundary": f"{CATEGORY_ORDER[j - 2]}|{CATEGORY_ORDER[j - 1]}",
               "n_levels": levels, "n_trials": int(g["n"].sum())}
        if levels < MIN_LEVELS_FOR_FIT:
            rec.update({"mu": np.nan, "sigma": np.nan, "lapse": np.nan,
                        "note": f"only {levels} stimulus levels; need >= {MIN_LEVELS_FOR_FIT}"})
            out.append(rec)
            continue
        f = fit_psychometric(g["x"], g["n"], g["k"])
        if f is None:
            rec.update({"mu": np.nan, "sigma": np.nan, "lapse": np.nan, "note": "fit failed"})
        else:
            rec.update(f)
            rec.update(bootstrap_psychometric(g["x"], g["n"], g["k"], n_boot=n_boot))
            rec["note"] = ""
        out.append(rec)
    fits = pd.DataFrame(out)
    if verbose:
        print("Psychometric fits (cumulative Gaussian with lapse):\n")
        cols = [c for c in ["boundary", "mu", "mu_lo", "mu_hi", "sigma", "sigma_lo",
                            "sigma_hi", "lapse", "n_levels", "n_trials", "note"]
                if c in fits.columns]
        print(fits[cols].round(4).to_string(index=False))
        print("\n  mu    = category boundary in deg VA (the 50% point, lapse-corrected)")
        print("  sigma = width of the transition; SMALLER = sharper discrimination")
        print("  lapse = stimulus-independent error rate (motor lapses, inattention)")
    return fits, tab


def plot_psychometric(fits, tab, stimulus_col=STIMULUS_COL):
    if tab.empty:
        return None
    splits = sorted(tab["split"].unique())
    fig, axes = plt.subplots(1, len(splits), figsize=(5.2 * len(splits), 4.2), squeeze=False)
    axes = axes.ravel()
    for ax, j in zip(axes, splits):
        g = tab[tab["split"] == j]
        row = fits[fits["split"] == j]
        se = np.sqrt(np.clip(g["p"] * (1 - g["p"]), 0, None) / np.maximum(g["n"], 1))
        ax.errorbar(g["x"], g["p"], yerr=1.96 * se, fmt="o", ms=6, capsize=3,
                    color="#31688e", label="observed")
        for _i, (_, r) in enumerate(g.sort_values("x").iterrows()):
            ax.annotate(f"{int(r['n'])}", (r["x"], r["p"]), textcoords="offset points",
                        xytext=(0, 8 if _i % 2 == 0 else 17), ha="center",
                        fontsize=6, color="grey")
        if not row.empty and np.isfinite(row["mu"].iloc[0]):
            mu, sig, lap = row["mu"].iloc[0], row["sigma"].iloc[0], row["lapse"].iloc[0]
            xx = np.linspace(g["x"].min() - 0.2, g["x"].max() + 0.2, 300)
            yy = lap / 2.0 + (1 - lap) * norm.cdf((xx - mu) / sig)
            ax.plot(xx, yy, "-", lw=2, color="#440154", label="fit")
            ax.axvline(mu, ls="--", lw=1.2, color="#fde725")
            lo, hi = row.get("mu_lo"), row.get("mu_hi")
            if lo is not None and np.isfinite(lo.iloc[0]):
                ax.axvspan(lo.iloc[0], hi.iloc[0], color="#fde725", alpha=0.25)
            ax.set_title(f"{row['boundary'].iloc[0]}\nmu={mu:.3f}  sigma={sig:.3f}  lapse={lap:.3f}",
                         fontsize=10)
            lo_cat, hi_cat = CATEGORY_ORDER[j - 2], CATEGORY_ORDER[j - 1]
            ax.axhspan(-0.03, 0.5, color=CATEGORY_COLORS.get(lo_cat, "#999999"), alpha=0.10)
            ax.axhspan(0.5, 1.03, color=CATEGORY_COLORS.get(hi_cat, "#999999"), alpha=0.10)
            ax.text(0.02, 0.06, lo_cat, transform=ax.transAxes, fontsize=8, color="#333333")
            ax.text(0.02, 0.92, hi_cat, transform=ax.transAxes, fontsize=8, color="#333333")
        else:
            ax.set_title(f"split {j}: not enough stimulus levels to fit", fontsize=10)
        ax.axhline(0.5, ls=":", lw=0.8, color="grey")
        ax.set_xlabel(f"{stimulus_col} (deg VA)")
        ax.set_ylabel(f"P(choose category >= {j})")
        ax.set_ylim(-0.03, 1.03)
        ax.legend(fontsize=8, loc="lower right")
    fig.suptitle("Psychometric functions: one ordinal split per category boundary", y=1.02)
    fig.tight_layout()
    fig.savefig("psychometric_functions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


# --- boundary distance and chronometric helpers (1.6) -----------------------
CHRONO_MEASURES = ["TotalTime_s", "DecisionTime_s", "ExecutionTime_s"]
NOMINAL_BOUNDARIES = [5.05, 6.40]
N_DISTANCE_BINS = 5


def boundary_distance(df, boundaries, stimulus_col=STIMULUS_COL):
    """Distance from each trial's bar length to the NEAREST category boundary.

    Small distance = an ambiguous stimulus sitting near the criterion; large
    distance = a prototypical one deep inside its category. This is the
    difficulty axis the chronometric and error analyses are built on, and it
    is derived from the fitted boundaries so it reflects where THIS subject
    actually put the criterion, not where the design assumed it.
    """
    x = pd.to_numeric(df[stimulus_col], errors="coerce").to_numpy(float)
    if not boundaries:
        return pd.Series(np.full(x.shape, np.nan), index=df.index)
    d = np.min(np.abs(x[:, None] - np.asarray(boundaries, float)[None, :]), axis=1)
    return pd.Series(d, index=df.index)


def chronometric_stats(df, measures=CHRONO_MEASURES, dist_col="boundary_dist"):
    rows = []
    for m in measures:
        if m not in df.columns:
            continue
        sub = df[[dist_col, m]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(sub) < 10 or sub[dist_col].nunique() < 3:
            rows.append({"measure": m, "n": len(sub), "spearman_rho": np.nan, "p": np.nan})
            continue
        rho, p = stats.spearmanr(sub[dist_col], sub[m])
        slope, intercept, r, p_lin, se = stats.linregress(sub[dist_col], sub[m])
        rows.append({"measure": m, "n": len(sub), "spearman_rho": rho, "p": p,
                     "slope_s_per_deg": slope, "slope_p": p_lin})
    return pd.DataFrame(rows)


def plot_chronometric(df, boundaries, measures=CHRONO_MEASURES,
                      dist_col="boundary_dist", stimulus_col=STIMULUS_COL,
                      n_bins=N_DISTANCE_BINS):
    measures = [m for m in measures if m in df.columns]
    if not measures:
        print("No timing columns available; chronometric plot skipped.")
        return None
    fig, axes = plt.subplots(2, len(measures), figsize=(4.8 * len(measures), 7.6), squeeze=False)
    d = df.copy()
    d["_bin"] = pd.qcut(d[dist_col], q=min(n_bins, d[dist_col].nunique()), duplicates="drop")
    for j, m in enumerate(measures):
        ax = axes[0][j]
        g = d.groupby(stimulus_col)[m].agg(["median", "count",
                                           lambda s: s.quantile(0.25),
                                           lambda s: s.quantile(0.75)])
        g.columns = ["median", "count", "q25", "q75"]
        g = g[g["count"] >= 3]
        ax.errorbar(g.index, g["median"],
                    yerr=[g["median"] - g["q25"], g["q75"] - g["median"]],
                    fmt="o-", ms=5, capsize=3, color="#31688e")
        for b in boundaries:
            ax.axvline(b, ls="--", lw=1.2, color="#fde725")
        ax.set_xlabel(f"{stimulus_col} (deg VA)")
        ax.set_ylabel(f"{m} (median, IQR)")
        ax.set_title(f"{m} by stimulus length\n(dashed = fitted boundary)", fontsize=9)

        ax = axes[1][j]
        gb = d.groupby("_bin", observed=True)[m].agg(["median", "count",
                                                      lambda s: s.quantile(0.25),
                                                      lambda s: s.quantile(0.75)])
        gb.columns = ["median", "count", "q25", "q75"]
        centres = [iv.mid for iv in gb.index]
        ax.errorbar(centres, gb["median"],
                    yerr=[gb["median"] - gb["q25"], gb["q75"] - gb["median"]],
                    fmt="s-", ms=6, capsize=3, color="#440154")
        ax.set_xlabel("distance to nearest boundary (deg VA)")
        ax.set_ylabel(f"{m} (median, IQR)")
        ax.set_title(f"{m} by difficulty\n(left = ambiguous, right = prototypical)", fontsize=9)
    fig.suptitle("Chronometric functions: does the subject slow down near the category boundary?", y=1.0)
    fig.tight_layout()
    fig.savefig("chronometric_functions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def plot_accuracy_by_difficulty(df, dist_col="boundary_dist", n_bins=N_DISTANCE_BINS):
    if "IsCorrect" not in df.columns:
        return None
    d = df[[dist_col, "IsCorrect"]].apply(pd.to_numeric, errors="coerce").dropna()
    if d.empty:
        return None
    d["_bin"] = pd.qcut(d[dist_col], q=min(n_bins, d[dist_col].nunique()), duplicates="drop")
    g = d.groupby("_bin", observed=True)["IsCorrect"].agg(["mean", "count"])
    se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
    fig, ax = plt.subplots(figsize=(5.6, 4.0))
    ax.errorbar([iv.mid for iv in g.index], g["mean"], yerr=1.96 * se,
                fmt="o-", ms=6, capsize=3, color="#21918c")
    ax.axhline(1.0 / 3.0, color=GRAY_LIGHT, lw=1, label="chance (3 categories)")
    ax.set_xlabel("distance to nearest boundary (deg VA)")
    ax.set_ylabel("P(correct)")
    ax.set_ylim(0, 1.02)
    ax.set_title("Accuracy by difficulty")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("accuracy_by_difficulty.png", dpi=FIGURE_DPI, bbox_inches="tight")
    rho, p = stats.spearmanr(d[dist_col], d["IsCorrect"])
    print(f"\nAccuracy vs distance-to-boundary: Spearman rho = {rho:+.3f}, p = {p:.2g} "
          f"(positive = more accurate on prototypical lengths, as expected).")
    return fig


# --- error classes (1.8) ----------------------------------------------------
ERROR_TYPE_LABELS = {
    0: "0 correct",
    1: "1 early exit / timeout (motor)",
    2: "2 wrong target (perceptual)",
    3: "3 hold break, strict (motor)",
}
PERCEPTUAL_ERRORS = {2}
MOTOR_ERRORS = {1, 3}


## 2. Across-session analysis

The single-session notebook describes one session. This section pools many.

Two things make pooling non-trivial, and both are handled explicitly:

- **Trials within a session are not independent.** Treating them as if they
  were is the easiest way to manufacture a learning effect that is not there,
  so session enters every model below as a grouping factor.
- **Sessions are not interchangeable.** They differ in stimulus set, in how
  many categories were live (`NumCategories`, `SessionMode`), and in length.
  `ConfigBarLengths.m` also changed the reduced sets' bar sizes at one point,
  so `BarSizeVA_deg` — the value actually shown — is what gets compared, never
  a category label.

Schema normalization runs per file, so a folder may freely mix engine
generations: a v8.19 session writing `ReactionTime_s` and a v8.20 one writing
`TotalTime_s` both land on the same canonical columns.

### 2.1. Pooling sessions and per-trial kinematics

Point `SESSION_GLOB` at the folder holding your sessions and run:

```python
SESSION_GLOB = "/content/sessions/trial_data_*.csv"
```

Each file keeps its identity in a `Session` column, and `SessionIndex` orders
them by date. That date is always parsed from the runTag in the file name
(`sessROM_<dd-mmm-yyyy>_<HH-MM>`), never taken from the file-system order or
the `Date` column, so `01-Sep` lands after `20-Aug` and two sessions on the
same day keep their order. The inventory table is meant to be read *before* any pooled
model: a pooled effect that is really a difference between two sessions'
designs shows up here as an obviously heterogeneous row, rather than as a
spurious slope three cells later.

The same cell then runs the 1.2 trajectory pipeline on every session and
attaches the per-trial kinematics to the pooled table. That adds a fourth
timing measure next to the task's own three clocks:

| measure | from | to | source |
|---|---|---|---|
| `DecisionTime_s` | go (target onset) | cursor leaves the centre circle | task |
| `ExecutionTime_s` | leaves the centre | reaches the target | task |
| `TotalTime_s` | go | reaches the target | task |
| `TakeoffTime_s` | go | the hand starts moving: `DecisionTime_s` minus `takeoff_lead_ms`, the time from the 5 % takeoff to the centre-circle exit | task + trajectory |

The takeoff is when the hand actually starts moving, which is before the
centre-circle crossing that `DecisionTime_s` reports. The lead between the two
is measured between two rows of the same trajectory, so it carries no
transport latency, and subtracting it from `DecisionTime_s` keeps the takeoff
on the same origin and clock as the decision time on both input sources. `TIMING_MEASURES` lists
the four, and every timing analysis below (2.2, 2.4, 2.5, 2.6) reads that list,
so the takeoff is treated exactly like the decision time.

For each session the cell also prints how many duplicate timestamps the
pipeline dropped, and builds `CLOCK_FIT_TABLE`: an ordinary least-squares fit
of `Time_ms` on `RZ2Idx` per session, with the implied write rate, its
deviation in ppm from $24414.0625/26$ Hz, the residuals of the online
timestamps about the fit, and the backward steps of `Time_ms` in index order.
Each session is classified as `rz2adc` or `usb` when its trajectory is read
(see 1.1 in the single-session notebook); the source is carried as
`InputSource` in the feature table, the pooled table and the session
inventory, and the clock fit only runs for rz2adc sessions (the others show
NaN). Sessions listed in `INVALID_SESSIONS` (0.1) are skipped when
the trial tables are pooled.


In [ ]:
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

SESSION_GLOB = globals().get("SESSION_GLOB") or "sessions/trial_data_*.csv"
RUN_TAG_RE = re.compile(r"trial_data_(.+)\.csv$")
MIN_TRIALS_PER_SESSION = 20


def _run_tag(path):
    m = RUN_TAG_RE.search(Path(path).name)
    return m.group(1) if m else Path(path).stem


def _session_start(tag):
    """Start time parsed from the runTag; session_timestamp from 0.1 when it
    is defined, the same regex inline otherwise, so this cell orders sessions
    by date on its own."""
    fn = globals().get("session_timestamp")
    if callable(fn):
        return fn(tag)
    m = re.search(r"_(\d{2}-[A-Za-z]{3}-\d{4})_(\d{2})-(\d{2})$", str(tag))
    if not m:
        return pd.NaT
    return pd.to_datetime(f"{m.group(1)} {m.group(2)}:{m.group(3)}",
                          format="%d-%b-%Y %H:%M", errors="coerce")


def _invalid_reason(tag):
    fn = globals().get("invalid_reason")
    if not callable(fn) or not globals().get("EXCLUDE_INVALID_SESSIONS", True):
        return None
    return fn(tag)


def load_sessions(pattern=SESSION_GLOB, min_trials=MIN_TRIALS_PER_SESSION):
    """Pool every trial_data_*.csv matching `pattern` into one long table.

    Each file keeps its own identity in a Session column (the runTag), because
    every across-session model below has to treat session as a grouping
    factor: trials within a session are not independent of each other, and
    pooling them as if they were is the single easiest way to manufacture a
    significant learning effect that is not there.

    normalize_trial_schema runs per file, so a folder may freely mix engine
    generations -- a v8.19 session writing ReactionTime_s and a v8.20 one
    writing TotalTime_s both land on the same canonical columns.
    """
    paths = sorted(glob.glob(pattern))
    if not paths:
        print(f"No files matched {pattern!r}.\n"
              f"Set SESSION_GLOB to the folder holding your sessions, e.g.\n"
              f"    SESSION_GLOB = '/content/sessions/trial_data_*.csv'\n"
              f"then re-run this cell.")
        return pd.DataFrame()
    frames, skipped = [], []
    for p in paths:
        reason = _invalid_reason(_run_tag(p))
        if reason:
            skipped.append((p, f"invalid session: {reason}"))
            continue
        try:
            td = normalize_trial_schema(pd.read_csv(p))
        except Exception as exc:
            skipped.append((p, str(exc)))
            continue
        if len(td) < min_trials:
            skipped.append((p, f"only {len(td)} trials"))
            continue
        td["Session"] = _run_tag(p)
        td["SourceFile"] = Path(p).name
        frames.append(td)
    if not frames:
        print(f"Matched {len(paths)} files but none were usable.")
        for p, why in skipped:
            print(f"  skipped {Path(p).name}: {why}")
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    # Chronological order ALWAYS comes from the runTag in the file name
    # (sessROM_<dd-mmm-yyyy>_<HH-MM>), never from the Date column inside the
    # CSV and never from the file-system order: alphabetically 01-Sep sorts
    # before 20-Aug, and the tag carries the start minute, so two sessions
    # on the same day still come out in the right order.
    out["SessionStart"] = out["Session"].map(_session_start)
    if out["SessionStart"].isna().any():
        bad = sorted(out.loc[out["SessionStart"].isna(), "Session"].unique())
        print(f"WARNING: no date in the runTag of {bad}; using the Date column for those.")
        if "Date" in out.columns:
            out["SessionStart"] = out["SessionStart"].fillna(
                pd.to_datetime(out["Date"], errors="coerce"))
    out["SessionDate"] = out["SessionStart"]
    order = (out.groupby("Session")["SessionStart"].min()
             .sort_values(kind="stable", na_position="last").index.tolist())
    # 1-based, like the 0.1 inventory: 1 is the earliest session in the window
    out["SessionIndex"] = out["Session"].map({s: i + 1 for i, s in enumerate(order)})
    out = out.sort_values(["SessionIndex", "Block", "TrialNumInBlock", "Attempt"],
                          kind="stable").reset_index(drop=True)
    print(f"Loaded {out['Session'].nunique()} sessions, {len(out)} trials total, "
          f"from {pattern!r}.")
    print("Session order (from the runTag date):")
    for s in order:
        print(f"  {out.loc[out['Session'] == s, 'SessionIndex'].iloc[0]:>3}  {s}  "
              f"{out.loc[out['Session'] == s, 'SessionStart'].iloc[0]}")
    for p, why in skipped:
        print(f"  skipped {Path(p).name}: {why}")
    return out


def session_inventory(df):
    """One row per session: what it contains and how it went.

    Read this before any pooled model. Sessions differ in stimulus set, in
    how many categories were live, and in length -- and a pooled effect that
    is really a difference between two sessions' designs will show up here as
    an obviously heterogeneous row rather than as a spurious slope later.
    """
    rows = []
    for s, g in df.groupby("Session", sort=False):
        rec = {"Session": s, "index": int(g["SessionIndex"].iloc[0]), "n_trials": len(g)}
        if "SessionDate" in g.columns:
            rec["date"] = g["SessionDate"].min()
        rec["n_blocks"] = int(g["Block"].nunique())
        rec["n_lengths"] = int(g[STIMULUS_COL].nunique())
        rec["pct_correct"] = 100.0 * float(pd.to_numeric(g["IsCorrect"], errors="coerce").mean())
        for c in TIMING_MEASURES:
            if c in g.columns:
                rec[f"median_{c}"] = float(pd.to_numeric(g[c], errors="coerce").median())
        if "NumCategories" in g.columns and g["NumCategories"].notna().any():
            rec["NumCategories"] = "/".join(str(int(v)) for v in sorted(g["NumCategories"].dropna().unique()))
        if "SessionMode" in g.columns and g["SessionMode"].notna().any():
            rec["mode"] = "/".join(sorted(g["SessionMode"].dropna().unique()))
        if "InputSource" in g.columns and g["InputSource"].notna().any():
            rec["input_source"] = "/".join(sorted(g["InputSource"].dropna().astype(str).unique()))
        e = pd.to_numeric(g["ErrorType"], errors="coerce")
        rec["pct_early_exit"] = 100.0 * float(e.isin(MOTOR_ERRORS).mean())
        rec["pct_wrong_target"] = 100.0 * float(e.isin(PERCEPTUAL_ERRORS).mean())
        rows.append(rec)
    return pd.DataFrame(rows).sort_values("index").reset_index(drop=True)


# ---------------------------------------------------------------------------
# Per-trial kinematics for every session, attached to the pooled trial table.
#
# trial_data_*.csv carries the task's own clocks (DecisionTime_s = go ->
# leave centre, ExecutionTime_s = leave centre -> target). The trajectory
# adds a kinematic clock. move_takeoff_ms is the first sample, walking back
# from the speed peak, still above MOVE_SPEED_FRAC (5%) of that peak. The lead
# takeoff_lead_ms = move_onset_ms - move_takeoff_ms is taken between two rows
# of the same trajectory, so transport latency cancels, and
# TakeoffTime_s = DecisionTime_s - lead puts the takeoff on the origin (go)
# and the clock of DecisionTime_s. It is a fourth timing measure below.
# ---------------------------------------------------------------------------
TIMING_MEASURES = ["TotalTime_s", "DecisionTime_s", "ExecutionTime_s", "TakeoffTime_s"]
FEATURE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)   # same movement window as 1.2
KINEMATIC_COLS = ["move_takeoff_ms", "takeoff_lead_ms", "TakeoffTime_s", "InputSource",
                  "n_samples", "fs_hz",
                  "peak_vel_cm", "mean_vel_cm", "median_vel_cm",
                  "peak_accel_cm", "mean_accel_cm", "median_accel_cm",
                  "path_length_cm", "straightness", "under_resolved"]


def _feature_inventory():
    """The 0.1 inventory (already in runTag date order), or a fresh scan."""
    inv = globals().get("SESSION_INVENTORY")
    if isinstance(inv, pd.DataFrame) and not inv.empty:
        return inv
    fn = globals().get("discover_sessions")
    return fn(verbose=False) if callable(fn) else pd.DataFrame()


def build_multi_session_features(inventory, min_trials=MIN_TRIALS_PER_SESSION, verbose=True):
    """Run the 1.2 feature pipeline (Hampel -> Kalman -> PCHIP -> Butterworth,
    then the per-trial kinematic summaries) on EVERY session of the inventory
    and stack the tables.

    Sessions are processed, indexed and labelled in the order of the date
    parsed from their runTag (sessROM_<dd-mmm-yyyy>_<HH-MM>), never in the
    order the file system lists them: alphabetically, 01-Sep sorts before
    20-Aug. SessionIndex is 1 for the earliest session in the window.
    """
    frames, skipped = [], []
    clock_rows = []
    inv = inventory.copy()
    if "SessionStart" not in inv.columns:
        inv["SessionStart"] = inv["Session"].map(_session_start)
    inv = inv.sort_values("SessionStart", kind="stable", na_position="last").reset_index(drop=True)
    for _, r in inv.iterrows():
        tag = r["Session"]
        traj = r.get("trajectory") or r.get("trajectory_movement")
        if not r.get("trial_data") or not traj:
            skipped.append((tag, "missing trial_data or trajectory export"))
            continue
        try:
            df = build_trial_features(traj, r["trial_data"], move_epochs=FEATURE_EPOCHS)
        except Exception as exc:
            skipped.append((tag, f"feature pipeline failed: {exc}"))
            continue
        if len(df) < min_trials:
            skipped.append((tag, f"only {len(df)} trials"))
            continue
        df["Session"] = tag
        df["SessionStart"] = r["SessionStart"]
        frames.append(df)
        source = df.attrs.get("input_source", "unknown")
        clock = (OfflineClockFit.from_csv(traj) if source == InputSourceDetector.RZ2
                 else OfflineClockFit([], [], source=Path(traj).name))
        clock_rows.append({"Session": tag, "InputSource": source, **clock.summary()})
        if verbose:
            n_kin = int(df["move_takeoff_ms"].notna().sum()) if "move_takeoff_ms" in df.columns else 0
            print(f"  {tag} [{source}]: {len(df)} trials, {n_kin} with kinematics, "
                  f"{df.attrs.get('n_unindexed_dropped', 0)} unindexed rows dropped, "
                  f"{df.attrs.get('n_duplicate_ts', 0)} duplicate timestamps dropped")
    for tag, why in skipped:
        print(f"  skipped {tag}: {why}")
    if clock_rows:
        globals()["CLOCK_FIT_TABLE"] = pd.DataFrame(clock_rows)
        print("\nOffline clock fit per session (OLS of Time_ms on RZ2Idx; NaN rate = no indexed rows):\n")
        print(globals()["CLOCK_FIT_TABLE"].round(3).to_string(index=False))
    if not frames:
        print("No session produced a feature table.")
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    if "TakeoffTime_s" not in out.columns:
        out["TakeoffTime_s"] = np.nan
    order = (out.groupby("Session")["SessionStart"].min()
             .sort_values(kind="stable", na_position="last").index.tolist())
    out["SessionIndex"] = out["Session"].map({s: i + 1 for i, s in enumerate(order)})
    stamp = out["SessionStart"].dt.strftime("%d-%b %H:%M").fillna("no date")
    out["SessionLabel"] = out["SessionIndex"].astype(str).str.zfill(2) + " | " + stamp
    out = out.sort_values(["SessionIndex", "Block", "TrialNumInBlock", "Attempt"],
                          kind="stable").reset_index(drop=True)
    return out


def attach_trial_kinematics(df, feats, cols=KINEMATIC_COLS):
    """Left-join the per-trial kinematics onto the pooled trial table on
    (Session, Block, TrialNumInBlock, Attempt). Trials whose session has no
    trajectory export, or whose movement was under-resolved, keep NaN."""
    if feats is None or feats.empty:
        print("No kinematics to attach; timing analyses will use the task clocks only.")
        return df
    keys = ["Session"] + ID_COLS
    cols = [c for c in cols if c in feats.columns]
    k = feats[keys + cols].drop_duplicates(keys)
    out = df.drop(columns=[c for c in cols if c in df.columns]).merge(k, on=keys, how="left")
    n = int(out["TakeoffTime_s"].notna().sum()) if "TakeoffTime_s" in out.columns else 0
    print(f"\nKinematics attached to {n}/{len(out)} trials ({100.0 * n / max(len(out), 1):.1f}%); "
          f"TakeoffTime_s joins {', '.join(TIMING_MEASURES[:-1])} as a timing measure.")
    none = sorted(set(out["Session"]) - set(feats["Session"]))
    if none:
        print(f"  sessions without kinematics: {none}")
    return out


def session_palette(labels):
    """One viridis colour per session, dark = earliest, bright = latest, so the
    date order is readable in every figure that colours by session."""
    cmap = plt.get_cmap("viridis")
    n = max(len(labels), 2)
    return {lab: cmap(i / (n - 1)) for i, lab in enumerate(labels)}


MULTI_DF = load_sessions()
if not MULTI_DF.empty:
    print("\nPer-trial kinematics from the trajectories (1.2 pipeline), session by session:")
    MULTI_FEATURES_DF = build_multi_session_features(_feature_inventory())
    MULTI_DF = attach_trial_kinematics(MULTI_DF, MULTI_FEATURES_DF)
    SESSION_TABLE = session_inventory(MULTI_DF)
    print("\nSession inventory:\n")
    print(SESSION_TABLE.round(3).to_string(index=False))
    SESSION_TABLE.to_csv("session_inventory.csv", index=False)
    MULTI_DF.to_csv("all_sessions_trials.csv", index=False)
    _sets = SESSION_TABLE["n_lengths"].unique()
    if len(_sets) > 1:
        print(f"\nHeterogeneous stimulus sets across sessions (distinct length counts: "
              f"{sorted(_sets)}). ConfigBarLengths.m also CHANGED the reduced sets' bar "
              f"sizes at one point, so before pooling reduced-set sessions check "
              f"BarSizeVA_deg, which records the value actually shown.")
else:
    SESSION_TABLE = pd.DataFrame()

### 2.2. Learning curves

Accuracy and timing per session (the four `TIMING_MEASURES`, movement
takeoff included), with Wilson intervals — they behave at the
extremes, where a session at 100% correct would otherwise get a zero-width
interval.

The trend is tested on **session means**, not on pooled trials. With ~150
trials per session a trial-level test would treat every trial as an independent
observation of *day* and massively overstate the evidence for a slope. Eight
sessions means eight observations.

The third panel checks within-session drift. These are rhesus monkeys working
for water, so satiation and fatigue push accuracy down through a session
independently of anything perceptual. If that panel slopes down, the
across-session trend needs reading with trial position as a covariate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

WINDOW_TRIALS = 40
NOMINAL_BOUNDARIES_MS = NOMINAL_BOUNDARIES if "NOMINAL_BOUNDARIES" in globals() else [5.05, 6.40]


def _wilson(k, n, z=1.96):
    """Wilson interval: behaves at the extremes, where a session at 100%
    correct would get a zero-width normal-approximation interval."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return c - h, c + h


def session_learning_table(df):
    """Accuracy and timing per session, with intervals and a trend test.

    The trend is tested on the SESSION means, not on the pooled trials: with
    ~150 trials per session, a trial-level test would treat every trial as an
    independent observation of 'day', which massively overstates the evidence
    for a learning slope. Eight sessions means eight observations.
    """
    rows = []
    for s, g in df.groupby("Session", sort=False):
        ok = pd.to_numeric(g["IsCorrect"], errors="coerce").dropna()
        k, n = int(ok.sum()), len(ok)
        lo, hi = _wilson(k, n)
        rec = {"Session": s, "index": int(g["SessionIndex"].iloc[0]),
               "n": n, "pct_correct": 100.0 * k / n if n else np.nan,
               "ci_lo": 100.0 * lo, "ci_hi": 100.0 * hi}
        for c in TIMING_MEASURES:
            if c in g.columns:
                v = pd.to_numeric(g[c], errors="coerce").dropna()
                rec[f"median_{c}"] = float(v.median()) if len(v) else np.nan
        rows.append(rec)
    tab = pd.DataFrame(rows).sort_values("index").reset_index(drop=True)
    tests = []
    for col in [c for c in tab.columns if c.startswith("median_")] + ["pct_correct"]:
        sub = tab[["index", col]].dropna()
        if len(sub) >= 4:
            rho, p = stats.spearmanr(sub["index"], sub[col])
            tests.append({"measure": col, "n_sessions": len(sub),
                          "spearman_rho": rho, "p": p})
    return tab, pd.DataFrame(tests)


def within_session_drift(df, window=WINDOW_TRIALS):
    """Accuracy against position WITHIN the session, pooled across sessions.

    Rhesus monkeys work for water, so late-session decline is a real and
    expected nuisance: satiation and fatigue both push accuracy down
    independently of anything perceptual. If this slopes down, the learning
    analysis above should be read on early-session trials, or with trial
    position as a covariate.
    """
    d = df.copy()
    d["trial_in_session"] = d.groupby("Session").cumcount()
    d["frac_through"] = d.groupby("Session")["trial_in_session"].transform(
        lambda s: s / max(s.max(), 1))
    d["decile"] = (d["frac_through"] * 10).clip(0, 9).astype(int)
    g = d.groupby("decile")["IsCorrect"].agg(["mean", "count"])
    rho, p = stats.spearmanr(d["frac_through"],
                             pd.to_numeric(d["IsCorrect"], errors="coerce"))
    return g, rho, p


def spread_labels(ys, min_gap):
    """Push direct-label y positions apart so they do not overlap, keeping
    their order; min_gap is in data units."""
    order = np.argsort(ys)
    out = np.array(ys, float)
    for a, b in zip(order[:-1], order[1:]):
        if out[b] - out[a] < min_gap:
            out[b] = out[a] + min_gap
    return out


TIMING_SHORT = {"TotalTime_s": "total", "DecisionTime_s": "decision",
                "ExecutionTime_s": "execution", "TakeoffTime_s": "takeoff"}


def plot_learning(tab, drift, tests=None):
    """Three panels, one message each. In the timing panel every measure is
    grey except the ones whose across-session trend is significant, which
    take the accent: the colour says where the story is."""
    sig = set()
    if tests is not None and not tests.empty:
        sig = set(tests.loc[tests["p"] < 0.05, "measure"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    ax = axes[0]
    ax.errorbar(tab["index"], tab["pct_correct"],
                yerr=[tab["pct_correct"] - tab["ci_lo"], tab["ci_hi"] - tab["pct_correct"]],
                fmt="o-", color=ACCENT, ecolor=GRAY, elinewidth=1)
    ax.axhline(100.0 / 3.0, color=GRAY_LIGHT, lw=1)
    ax.text(tab["index"].max(), 100.0 / 3.0 + 1.5, "chance (3 categories)",
            ha="right", va="bottom", fontsize=8, color=GRAY)
    ax.set_xlabel("session (date order)")
    ax.set_ylabel("% correct (Wilson 95% CI)")
    ax.set_title("Accuracy across sessions", fontsize=10)
    ax = axes[1]
    x_end = tab["index"].max()
    cols = [f"median_{m}" for m in TIMING_MEASURES
            if f"median_{m}" in tab.columns and tab[f"median_{m}"].notna().any()]
    y_all = tab[cols].to_numpy(float)
    label_y = spread_labels([tab[c].iloc[-1] for c in cols],
                            0.05 * (np.nanmax(y_all) - np.nanmin(y_all)))
    for c, ly in zip(cols, label_y):
        m = c[len("median_"):]
        hi = c in sig
        ax.plot(tab["index"], tab[c], "o-", color=ACCENT if hi else GRAY,
                lw=1.8 if hi else 1.2, ms=4 if hi else 3, alpha=1 if hi else 0.9)
        ax.text(x_end + 0.25, ly, TIMING_SHORT.get(m, m) + (" *" if hi else ""),
                va="center", fontsize=8, color=ACCENT if hi else GRAY)
    ax.set_xlim(tab["index"].min() - 0.5, x_end + 2.5)
    ax.set_xlabel("session (date order)")
    ax.set_ylabel("median (s)")
    ax.set_title("Timing across sessions\n(colour = significant trend, * p < 0.05)", fontsize=10)
    ax = axes[2]
    g = drift[0]
    se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["count"])
    ax.errorbar(g.index / 10.0 + 0.05, 100 * g["mean"], yerr=196 * se,
                fmt="o-", color=ACCENT, ecolor=GRAY, elinewidth=1)
    ax.set_xlabel("position through the session (deciles)")
    ax.set_ylabel("% correct")
    ax.set_title(f"Within-session drift\nSpearman rho = {drift[1]:+.3f}, p = {drift[2]:.3g}",
                 fontsize=10)
    fig.suptitle("Across-session learning and within-session fatigue", y=1.03, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("learning_curves.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


if not MULTI_DF.empty:
    LEARNING_TABLE, LEARNING_TESTS = session_learning_table(MULTI_DF)
    print("Per-session performance:\n")
    print(LEARNING_TABLE.round(3).to_string(index=False))
    print("\nTrend across sessions (Spearman on session means, n = one point per session):\n")
    print(LEARNING_TESTS.round(4).to_string(index=False))
    DRIFT = within_session_drift(MULTI_DF)
    print(f"\nWithin-session drift: accuracy vs position through the session, "
          f"Spearman rho = {DRIFT[1]:+.4f}, p = {DRIFT[2]:.4g}")
    if DRIFT[2] < 0.05 and DRIFT[1] < 0:
        print("  Accuracy DECLINES through a session. Treat the across-session trend "
              "with care: sessions of different lengths carry different amounts of "
              "this decline.")
    plot_learning(LEARNING_TABLE, DRIFT, LEARNING_TESTS)
    plt.show()
    LEARNING_TABLE.to_csv("learning_curves.csv", index=False)

### 2.3. Psychometric parameters across sessions, and mixed-effects models

Refitting the psychometric curves session by session turns a descriptive
learning curve into a mechanistic one. Rising accuracy can come from three
different changes, and only separate fits tell them apart:

- **mu moves** — the subject relocated the category boundary.
- **sigma shrinks** — genuinely sharper discrimination.
- **lapse falls** — better engagement, unchanged perception.

The trial-level model is then fitted twice, because the two answer different
questions. **GEE** gives the population-averaged effect with standard errors
that account for within-session correlation — the number to quote for "does
difficulty predict accuracy". The **mixed GLM** adds a random intercept per
session, and its variance component says how much sessions differ beyond the
fixed effects.

Difficulty in these models is distance to the **design** boundaries, not the
fitted ones: using fitted boundaries would put a quantity estimated from these
same choices on the right-hand side of a model of those choices.

In [ ]:
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PER_SESSION_BOOTSTRAP = 200


def per_session_psychometrics(df, n_boot=PER_SESSION_BOOTSTRAP):
    """Refit the psychometric curves separately in every session.

    This is where learning becomes mechanistic rather than descriptive. A
    rising accuracy curve can come from three different changes, and only
    separate fits tell them apart:
      mu    moving   -> the subject relocated the category boundary
      sigma shrinking-> genuinely sharper discrimination
      lapse falling  -> better engagement, unchanged perception
    """
    rows = []
    for s, g in df.groupby("Session", sort=False):
        fits, _ = fit_all_boundaries(g, n_boot=n_boot, verbose=False)
        if fits.empty:
            continue
        for _, r in fits.iterrows():
            rows.append({"Session": s, "index": int(g["SessionIndex"].iloc[0]),
                         "boundary": r.get("boundary"), "mu": r.get("mu"),
                         "sigma": r.get("sigma"), "lapse": r.get("lapse"),
                         "mu_lo": r.get("mu_lo"), "mu_hi": r.get("mu_hi"),
                         "n_trials": r.get("n_trials"), "note": r.get("note", "")})
    return pd.DataFrame(rows)


def plot_per_session_params(par):
    if par.empty:
        return None
    bounds = [b for b in par["boundary"].dropna().unique()]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    colors = [ACCENT, ACCENT_2, GRAY]
    for i, b in enumerate(bounds):
        sub = par[par["boundary"] == b].sort_values("index")
        c = colors[i % len(colors)]
        ax = axes[0]
        yerr = None
        if sub["mu_lo"].notna().all() and sub["mu_hi"].notna().all():
            yerr = [sub["mu"] - sub["mu_lo"], sub["mu_hi"] - sub["mu"]]
        ax.errorbar(sub["index"], sub["mu"], yerr=yerr, fmt="o-", ms=5,
                    color=c, ecolor=GRAY, elinewidth=1, label=b)
        axes[1].plot(sub["index"], sub["sigma"], "o-", ms=5, color=c, label=b)
        axes[2].plot(sub["index"], sub["lapse"], "o-", ms=5, color=c, label=b)
    for ax, ttl, yl in ((axes[0], "Boundary location (mu)", "deg VA"),
                        (axes[1], "Discrimination width (sigma)", "deg VA"),
                        (axes[2], "Lapse rate", "proportion")):
        ax.set_xlabel("session (date order)"); ax.set_ylabel(yl)
        ax.set_title(ttl, fontsize=10); ax.legend(fontsize=7)
    axes[1].set_title("Discrimination width (sigma)\nDOWN = sharper", fontsize=10)
    fig.suptitle("Psychometric parameters across sessions: what actually changed", y=1.03, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("psychometric_across_sessions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def mixed_effects_accuracy(df):
    """Trial-level accuracy model with session as a grouping factor.

    Two fits, because they answer different questions and disagree in
    informative ways:

      GEE (population-averaged) -- the average effect across sessions, with
      standard errors that account for trials being correlated within a
      session. This is the number to quote for 'does difficulty predict
      accuracy'.

      Mixed GLM (subject-specific) -- adds a random intercept per session,
      so the variance component says how much sessions differ from each other
      on top of the fixed effects.

    Difficulty is distance to the DESIGN boundaries, not to the fitted ones:
    using fitted boundaries here would put a quantity estimated from these
    same choices on the right-hand side of a model of those choices.
    """
    d = df.copy()
    d["IsCorrect"] = pd.to_numeric(d["IsCorrect"], errors="coerce")
    d["difficulty"] = boundary_distance(d, NOMINAL_BOUNDARIES)
    d["session_idx"] = d["SessionIndex"].astype(float)
    d = d[["IsCorrect", "difficulty", "session_idx", "Session"]].dropna()
    if d["Session"].nunique() < 3 or len(d) < 100:
        print("Too few sessions or trials for a mixed model; skipped.")
        return None, None
    d["session_c"] = d["session_idx"] - d["session_idx"].mean()
    formula = "IsCorrect ~ difficulty + session_c"
    gee = smf.gee(formula, groups="Session", data=d,
                  family=sm.families.Binomial(),
                  cov_struct=sm.cov_struct.Exchangeable()).fit()
    print("GEE (population-averaged logistic, exchangeable within session):\n")
    print(gee.summary().tables[1])
    print(f"\n  Interpretation, on the log-odds scale:")
    for name in ("difficulty", "session_c"):
        if name in gee.params.index:
            b, p = gee.params[name], gee.pvalues[name]
            what = ("each extra deg VA away from the boundary" if name == "difficulty"
                    else "each additional session")
            direction = "raises" if b > 0 else "lowers"
            sig = "" if p < 0.05 else "  (not significant)"
            print(f"    {what} {direction} the odds of a correct trial by a factor of "
                  f"{np.exp(b):.3f}  (p = {p:.3g}){sig}")
    mixed = None
    try:
        from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
        mixed = BinomialBayesMixedGLM.from_formula(
            formula, {"session": "0 + C(Session)"}, d).fit_vb(verbose=False)
        sd = float(np.exp(mixed.vcp_mean[0]))
        print(f"\nMixed GLM with a random intercept per session: "
              f"between-session SD on the log-odds scale = {sd:.3f}.")
        print(f"  Sessions differ from one another by about a factor of {np.exp(sd):.2f} "
              f"in the odds of a correct trial, beyond the fixed effects above. A large "
              f"value here is the reason the pooled analysis needs the grouping factor.")
    except Exception as exc:
        print(f"\nRandom-intercept fit unavailable ({type(exc).__name__}); "
              f"the GEE result above already accounts for within-session correlation.")
    return gee, mixed


if not MULTI_DF.empty:
    PER_SESSION_PARAMS = per_session_psychometrics(MULTI_DF)
    if not PER_SESSION_PARAMS.empty:
        print("Psychometric parameters per session:\n")
        print(PER_SESSION_PARAMS.round(4).to_string(index=False))
        PER_SESSION_PARAMS.to_csv("psychometric_across_sessions.csv", index=False)
        from scipy import stats as _st
        print("\nDo the parameters move systematically across sessions?\n")
        _rows = []
        for b, g in PER_SESSION_PARAMS.groupby("boundary"):
            for param in ("mu", "sigma", "lapse"):
                sub = g[["index", param]].dropna()
                if len(sub) >= 4:
                    rho, p = _st.spearmanr(sub["index"], sub[param])
                    _rows.append({"boundary": b, "parameter": param,
                                  "n_sessions": len(sub), "spearman_rho": rho, "p": p})
        _tr = pd.DataFrame(_rows)
        print(_tr.round(4).to_string(index=False))
        print("\n  sigma falling across sessions = the subject is discriminating more")
        print("  finely. mu drifting = the boundary itself is moving. lapse falling =")
        print("  better engagement rather than better perception.")
        plot_per_session_params(PER_SESSION_PARAMS)
        plt.show()
    print()
    GEE_FIT, MIXED_FIT = mixed_effects_accuracy(MULTI_DF)

### 2.4. Per-trial features across sessions

`MULTI_FEATURES_DF`, built in 2.1 by running the 1.2 feature engineering
(Hampel, Kalman + RTS, PCHIP, Butterworth, then the per-trial kinematic
summaries) on **every session in the window**, carries `Session`,
`SessionStart` and `SessionIndex`; `MULTI_ANALYSIS_DF` is the same without the
early exits, as in 1.2. The movement takeoff (`move_takeoff_ms`) is one of the
features and takes part in every figure and test here.

Sessions are processed, numbered and coloured in the order of the date parsed
from their runTag. In every figure of this and the next section the viridis
scale runs dark for the earliest session to bright for the latest.

Then the 1.2 figures and tests on the pooled trials: the pairplot (by category
and, a second time, by session), the violins by chosen target and by
correctness with their Kruskal-Wallis tests, the correct-vs-incorrect
Mann-Whitney table, the accuracy breakdown and the correlation heatmap.
Pooled means every trial of every session with equal weight, so read these
together with 2.5, which asks how much of what they show is a between-session
difference.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SESSION_FEATURES = ["DecisionTime_s", "ExecutionTime_s", "TotalTime_s",
                    "n_samples", "move_takeoff_ms",
                    "peak_vel_cm", "mean_vel_cm", "peak_accel_cm", "mean_accel_cm",
                    "path_length_cm", "straightness",
                    "hold_dur_ms", "hold_max_speed_cm", "hold_excursion_px"]
SESSION_PAIRPLOT_FEATURES = ["BarSizeVA_deg", "DecisionTime_s", "ExecutionTime_s", "move_takeoff_ms",
                             "peak_vel_cm", "peak_accel_cm", "path_length_cm", "straightness"]


def pairplot_by_session(df, features=SESSION_PAIRPLOT_FEATURES, labels=None, palette=None):
    """The 1.2 pairplot, coloured by session in date order instead of by category."""
    cols = [c for c in features if c in df.columns]
    sub = df.dropna(subset=cols).copy()
    labels = labels or sorted(sub["SessionLabel"].unique())
    print(f"Pairplot by session on {len(sub)}/{len(df)} trials.")
    g = sns.pairplot(sub, vars=cols, hue="SessionLabel", hue_order=labels,
                     palette=palette or session_palette(labels), corner=True,
                     diag_kind="kde", plot_kws=dict(s=14, alpha=0.5, edgecolor="none"))
    g.figure.suptitle("Per-trial feature pairplot, coloured by session (dark = earliest)", y=1.02)
    g.figure.savefig("trial_pairplot_by_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return g


if "MULTI_FEATURES_DF" not in globals() or MULTI_FEATURES_DF.empty:
    # 2.1 normally builds this; rebuild here if 2.1 was skipped
    MULTI_INV = _feature_inventory()
    MULTI_FEATURES_DF = (build_multi_session_features(MULTI_INV)
                        if not MULTI_INV.empty else pd.DataFrame())
if MULTI_FEATURES_DF.empty:
    print("No sessions with trajectories in the window; run 0.1 (or set OUTPUTS_DIR) and re-run.")
    MULTI_ANALYSIS_DF = pd.DataFrame()
    SESSION_LABELS, SESSION_PALETTE = [], {}
else:
    MULTI_FEATURES_DF.to_csv("trial_features_all_sessions.csv", index=False)
    MULTI_ANALYSIS_DF = drop_early_exits(MULTI_FEATURES_DF)
    MULTI_ANALYSIS_DF.to_csv("trial_features_all_sessions_no_early_exits.csv", index=False)
    SESSION_LABELS = (MULTI_ANALYSIS_DF.drop_duplicates("Session")
                    .sort_values("SessionIndex")["SessionLabel"].tolist())
    SESSION_PALETTE = session_palette(SESSION_LABELS)
    print(f"\nFeature table: {len(MULTI_FEATURES_DF)} trials x {MULTI_FEATURES_DF.shape[1]} columns "
        f"from {MULTI_FEATURES_DF['Session'].nunique()} sessions, in date order:")
    print(MULTI_FEATURES_DF.groupby(["SessionIndex", "Session"], sort=True)
        .agg(start=("SessionStart", "first"), trials=("IsCorrect", "size"),
            accuracy=("IsCorrect", "mean")).round(3).to_string())

    # The same figures and tests as 1.2, on every session pooled.
    pairplot_trials(MULTI_ANALYSIS_DF, features=SESSION_PAIRPLOT_FEATURES)
    plt.show()
    pairplot_by_session(MULTI_ANALYSIS_DF, labels=SESSION_LABELS, palette=SESSION_PALETTE)
    plt.show()
    violin_by_group(MULTI_ANALYSIS_DF, group_col=VIOLIN_GROUP,
                    savename="violin_by_chosentarget_all_sessions.png")
    plt.show()
    violin_by_group(MULTI_ANALYSIS_DF, group_col=CORRECTNESS_GROUP,
                    savename="violin_by_iscorrect_all_sessions.png")
    plt.show()
    MULTI_CORRECTNESS_STATS = correct_vs_incorrect(MULTI_ANALYSIS_DF)
    plt.show()
    MULTI_ACCURACY_BREAKDOWN = accuracy_breakdown(MULTI_ANALYSIS_DF)
    plt.show()
    MULTI_CORR = correlation_heatmap(MULTI_ANALYSIS_DF.drop(columns=["SessionIndex"]))
    plt.gca().figure.savefig("trial_corr_heatmap_all_sessions.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()


### 2.5. Feature statistics by session

Per feature, two questions with sessions in date order:

- **Does it differ between sessions?** Kruskal-Wallis across sessions with
  epsilon-squared as the effect size, FDR-corrected across features.
- **Does it drift with date?** Spearman correlation of the per-session median
  against session order, tested on session medians (one observation per
  session, as in 2.2), FDR-corrected.

Figures: hollow violins per session for every feature with the medians joined
by a line; a sessions x features heatmap of z-scored medians; the correlation
heatmap of 1.2 repeated per session, with the mean and the spread of each
pairwise correlation across sessions (a stable pair is a property of the
movement, an unstable one of that day); and the correct-vs-incorrect
rank-biserial effect repeated per session.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

SESSION_STATS_ALPHA = 0.05


def feature_session_stats(df, features=SESSION_FEATURES, alpha=SESSION_STATS_ALPHA):
    """Two questions per feature, FDR-corrected across features.

    - Does it DIFFER between sessions at all? Kruskal-Wallis over sessions,
      with epsilon-squared as the effect size (fraction of rank variance
      explained by session).
    - Does it DRIFT with date? Spearman correlation of the per-session MEDIAN
      against session order. Tested on session medians, not on trials, for
      the reason given in 2.2: ten sessions are ten observations of "day".
    """
    feats = [f for f in features if f in df.columns]
    rows = []
    for f in feats:
        sub = df[["SessionIndex", f]].apply(pd.to_numeric, errors="coerce").dropna()
        groups = [g[f].to_numpy() for _, g in sub.groupby("SessionIndex") if len(g) >= 3]
        H = p = eps2 = np.nan
        k, n = len(groups), int(sum(a.size for a in groups))
        if k >= 2 and sub[f].nunique() > 1:
            H, p = stats.kruskal(*groups)
            eps2 = (H - k + 1) / (n - k) if n > k else np.nan
        med = sub.groupby("SessionIndex")[f].median()
        rho = p_rho = np.nan
        if len(med) >= 3 and med.nunique() > 1:
            rho, p_rho = stats.spearmanr(med.index.to_numpy(), med.to_numpy())
        rows.append({"feature": f, "n_sessions": k, "n_trials": n,
                     "kruskal_H": H, "kruskal_p": p, "epsilon_sq": eps2,
                     "trend_rho": rho, "trend_p": p_rho,
                     "median_first": float(med.iloc[0]) if len(med) else np.nan,
                     "median_last": float(med.iloc[-1]) if len(med) else np.nan})
    res = pd.DataFrame(rows)
    res["q_fdr"] = _bh_fdr(res["kruskal_p"].to_numpy())
    res["trend_q_fdr"] = _bh_fdr(res["trend_p"].to_numpy())
    res[f"differs(FDR<{alpha})"] = np.where(res["q_fdr"] < alpha, "yes", "no")
    res[f"drifts(FDR<{alpha})"] = np.where(res["trend_q_fdr"] < alpha, "yes", "no")
    res = res.reindex(res["kruskal_p"].fillna(1).sort_values().index).reset_index(drop=True)
    print("Per-feature statistics across sessions (sessions in date order):\n")
    print(res.round(4).to_string(index=False))
    print(f"\n{int((res['q_fdr'] < alpha).sum())}/{len(res)} features differ between sessions "
          f"and {int((res['trend_q_fdr'] < alpha).sum())}/{len(res)} drift monotonically with "
          f"date at FDR < {alpha}.")
    print("  epsilon_sq is the share of the feature's rank variance that session explains;")
    print("  trend_rho > 0 means the session median GROWS from the earliest to the latest session.")
    return res


def feature_by_session_table(df, features=SESSION_FEATURES):
    """Median [IQR] of every feature in every session, sessions in date order."""
    feats = [f for f in features if f in df.columns]
    g = df.groupby(["SessionIndex", "SessionLabel"], sort=True)[feats]
    med, q25, q75 = g.median(), g.quantile(0.25), g.quantile(0.75)
    tab = med.copy().astype(object)
    for f in feats:
        tab[f] = [f"{m:.3g} [{a:.3g}-{b:.3g}]" for m, a, b in zip(med[f], q25[f], q75[f])]
    tab.insert(0, "n", g.size())
    return tab, med, q25, q75


def _short_session_labels(labels):
    """'01 | 20-Aug 14:44' -> '01\\n20-Aug': the index carries the order, the
    day is enough to recognise the session, the minute is clutter on an axis."""
    out = []
    for lab in labels:
        idx, _, stamp = lab.partition(" | ")
        out.append(f"{idx}\n{stamp.split(' ')[0]}")
    return out


def plot_features_by_session(df, features=SESSION_FEATURES, labels=None, palette=None):
    """One panel per feature: grey hollow violins per session in date order,
    with the per-session median joined by a dark line. The sessions are
    already ordered on the axis, so colouring them would only repeat that;
    the one thing to read is the median line, so it is the only dark mark."""
    feats = [f for f in features if f in df.columns]
    labels = labels or sorted(df["SessionLabel"].unique())
    palette = palette or session_palette(labels)
    ncol = 3
    nrow = int(np.ceil(len(feats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for i, f in enumerate(feats):
        ax = axes[i]
        sub = df[["SessionLabel", f]].dropna()
        sns.stripplot(x="SessionLabel", y=f, data=sub, order=labels, ax=ax,
                      color=GRAY_LIGHT, size=1.6, jitter=0.28, alpha=0.6,
                      linewidth=0, zorder=0.5)
        sns.violinplot(x="SessionLabel", y=f, data=sub, order=labels, ax=ax,
                       color=GRAY, inner="box", cut=0, density_norm="width", fill=False,
                       linewidth=1.0, inner_kws=dict(box_width=3, whis_width=0.8, color=GRAY))
        med = sub.groupby("SessionLabel")[f].median().reindex(labels)
        ax.plot(range(len(labels)), med.to_numpy(), color=INK, lw=1.4,
                marker="o", ms=3, zorder=5)
        ax.set_title(f, fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(_short_session_labels(labels), fontsize=6)
        ax.tick_params(axis="y", labelsize=7)
    for j in range(len(feats), len(axes)):
        axes[j].axis("off")
    fig.suptitle("Per-trial features by session (earliest on the left; dark line = session median)",
                 y=1.0, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("features_by_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def session_feature_heatmap(df, features=SESSION_FEATURES, labels=None):
    """Sessions x features heatmap. Colour is the z-score of each session's
    median across sessions (which sessions are unusually high or low on each
    feature); the annotation is the raw median so the units stay visible."""
    feats = [f for f in features if f in df.columns]
    labels = labels or sorted(df["SessionLabel"].unique())
    med = df.groupby("SessionLabel")[feats].median().reindex(labels)
    sd = med.std(ddof=0).replace(0, np.nan)
    z = (med - med.mean()) / sd
    fig, ax = plt.subplots(figsize=(0.95 * len(feats) + 3, 0.45 * len(labels) + 2))
    sns.heatmap(z, annot=med.to_numpy(), fmt=".2f", cmap="vlag", center=0, vmin=-2.5, vmax=2.5,
                cbar_kws={"shrink": 0.8, "label": "z of session median"}, ax=ax,
                annot_kws={"size": 6}, linewidths=0.4, linecolor="white")
    ax.set_xlabel("")
    ax.set_ylabel("session (date order)")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_title("Session medians per feature (colour = z across sessions, text = median)")
    fig.tight_layout()
    fig.savefig("session_feature_heatmap.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return med, z


def per_session_correlation(df, features=SESSION_FEATURES, labels=None, method=CORRELATION_METHOD):
    """One correlation heatmap per session (date order), then the mean and the
    spread of every pairwise correlation across sessions. A pair whose
    correlation is stable is a property of the movement; one that swings
    between sessions is a property of that day's session."""
    feats = [f for f in features if f in df.columns]
    labels = labels or sorted(df["SessionLabel"].unique())
    mats = {}
    for lab in labels:
        sub = df.loc[df["SessionLabel"] == lab, feats]
        sub = sub.loc[:, sub.nunique(dropna=True) > 1]
        mats[lab] = sub.corr(method=method).reindex(index=feats, columns=feats)
    n = len(labels)
    ncol = min(4, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.3 * nrow), squeeze=False)
    axes = axes.ravel()
    for ax, lab in zip(axes, labels):
        sns.heatmap(mats[lab], vmin=-1, vmax=1, center=0, cmap="RdBu_r", square=True, cbar=False, ax=ax,
                    xticklabels=feats, yticklabels=feats)
        ax.set_title(lab, fontsize=8)
        ax.tick_params(axis="both", labelsize=5)
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(f"{method.capitalize()} correlation of per-trial features, one panel per session",
                 y=1.0, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("trial_corr_heatmap_per_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()

    stack = np.stack([m.to_numpy(dtype=float) for m in mats.values()])
    mean = pd.DataFrame(np.nanmean(stack, axis=0), index=feats, columns=feats)
    spread = pd.DataFrame(np.nanstd(stack, axis=0), index=feats, columns=feats)
    fig, axes = plt.subplots(1, 2, figsize=(0.55 * len(feats) * 2 + 6, 0.55 * len(feats) + 2))
    sns.heatmap(mean, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True,
                ax=axes[0], annot_kws={"size": 6}, cbar_kws={"shrink": 0.7})
    axes[0].set_title("mean correlation across sessions", fontsize=9)
    sns.heatmap(spread, annot=True, fmt=".2f", cmap="Blues", vmin=0, square=True,
                ax=axes[1], annot_kws={"size": 6}, cbar_kws={"shrink": 0.7})
    axes[1].set_title("SD of the correlation across sessions (high = unstable pair)", fontsize=9)
    for ax in axes:
        ax.tick_params(axis="both", labelsize=6)
    fig.tight_layout()
    fig.savefig("trial_corr_heatmap_session_stability.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return mean, spread


def correctness_effect_by_session(df, features=SESSION_FEATURES, labels=None, min_n=5):
    """The 1.2 correct-vs-incorrect comparison repeated in every session: a
    sessions x features heatmap of the rank-biserial effect (> 0 = larger on
    correct trials), so a feature that separates outcomes only on some days
    is visible as a row-wise pattern rather than averaged away."""
    feats = [f for f in features if f in df.columns]
    labels = labels or sorted(df["SessionLabel"].unique())
    d = df.copy()
    if "ChosenTarget" in d.columns:
        d = d[pd.to_numeric(d["ChosenTarget"], errors="coerce") >= 1]
    d["IsCorrect"] = pd.to_numeric(d["IsCorrect"], errors="coerce")
    eff = pd.DataFrame(np.nan, index=labels, columns=feats)
    pval = eff.copy()
    for lab in labels:
        g = d[d["SessionLabel"] == lab]
        for f in feats:
            s = g[["IsCorrect", f]].apply(pd.to_numeric, errors="coerce").dropna()
            a = s.loc[s["IsCorrect"] == 1, f].to_numpy()
            b = s.loc[s["IsCorrect"] == 0, f].to_numpy()
            if a.size < min_n or b.size < min_n or s[f].nunique() < 2:
                continue
            U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
            eff.loc[lab, f] = 2.0 * U / (a.size * b.size) - 1.0
            pval.loc[lab, f] = p
    if eff.notna().sum().sum() == 0:
        print("Too few incorrect trials per session for a per-session correct/incorrect comparison.")
        return eff, pval
    fig, ax = plt.subplots(figsize=(0.95 * len(feats) + 3, 0.45 * len(labels) + 2))
    sns.heatmap(eff.astype(float), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
                cbar_kws={"shrink": 0.8, "label": "rank-biserial (correct - incorrect)"}, ax=ax,
                annot_kws={"size": 6}, linewidths=0.4, linecolor="white")
    ax.set_ylabel("session (date order)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_title(f"Correct vs incorrect per session (blank = fewer than {min_n} trials on one side)")
    fig.tight_layout()
    fig.savefig("correct_vs_incorrect_by_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    consistent = ((eff > 0).sum() == eff.notna().sum()) | ((eff < 0).sum() == eff.notna().sum())
    consistent = consistent[eff.notna().sum() >= 3]
    if consistent.any():
        print("Features whose correct/incorrect difference has the SAME sign in every session "
              f"with enough trials: {consistent[consistent].index.tolist()}")
    return eff, pval


def numeric_features(df, features=SESSION_FEATURES):
    """Coerce the feature columns to numbers and keep only those with data.
    hold_* are empty when the movement window (FEATURE_EPOCHS) excludes the
    target hold, exactly as in 1.2, and an all-NaN column would otherwise
    break the quantiles and leave blank panels."""
    d = df.copy()
    keep, empty = [], []
    for f in features:
        if f not in d.columns:
            continue
        d[f] = pd.to_numeric(d[f], errors="coerce")
        (keep if d[f].notna().any() else empty).append(f)
    if empty:
        print(f"Features with no data in this window, skipped: {empty}")
    return d, keep


if MULTI_ANALYSIS_DF.empty:
    print("No pooled feature table (2.4 found no sessions); nothing to compare.")
else:
    _D, SESSION_FEATURES_AVAILABLE = numeric_features(MULTI_ANALYSIS_DF)
    SESSION_FEATURE_STATS = feature_session_stats(_D, features=SESSION_FEATURES_AVAILABLE)
    SESSION_FEATURE_STATS.to_csv("feature_stats_across_sessions.csv", index=False)

    SESSION_FEATURE_TABLE, SESSION_MEDIANS, _q25, _q75 = feature_by_session_table(_D, features=SESSION_FEATURES_AVAILABLE)
    print("\nMedian [IQR] per session, in date order:\n")
    print(SESSION_FEATURE_TABLE.to_string())
    SESSION_MEDIANS.to_csv("feature_medians_by_session.csv")

    plot_features_by_session(_D, features=SESSION_FEATURES_AVAILABLE, labels=SESSION_LABELS, palette=SESSION_PALETTE)
    plt.show()
    session_feature_heatmap(_D, features=SESSION_FEATURES_AVAILABLE, labels=SESSION_LABELS)
    plt.show()
    SESSION_CORR_MEAN, SESSION_CORR_SD = per_session_correlation(_D, features=SESSION_FEATURES_AVAILABLE, labels=SESSION_LABELS)
    plt.show()
    SESSION_CORRECTNESS_EFFECT, _ = correctness_effect_by_session(_D, features=SESSION_FEATURES_AVAILABLE, labels=SESSION_LABELS)
    plt.show()


### 2.6. Timing measures across sessions

Every timing measure in `TIMING_MEASURES` (total, decision, execution and the
movement takeoff) gets the same treatment, so the takeoff can be read side by
side with the decision time:

- **Takeoff vs decision clock.** Per session: medians of both, the gap between
  them (time the hand is already moving before it crosses the centre circle),
  and their rank correlation. Since `TakeoffTime_s` is `DecisionTime_s` minus
  `takeoff_lead_ms`, the gap is the lead itself, measured within one
  trajectory, so a gap that changes across sessions reflects when the hand
  starts relative to the centre exit rather than a difference between clocks.
- **Chronometric relation per session** (1.6, repeated per session): Spearman
  rho of each measure against distance to the nearest design boundary, plus the
  per-session median curves against bar length. Whether the slowing on
  ambiguous lengths lives in the takeoff (the decision is slow to start) or in
  the execution (the reach itself) is the question.
- **GEE per measure**, the timing counterpart of the accuracy model in 2.3:
  `log(measure) ~ difficulty + session`, grouped by session, so the coefficients
  read as a percent change per deg VA from the boundary and per session.
- **Post-error slowing per session** (1.7, repeated per session): median after
  an error minus median after a correct trial, predecessor taken within the
  block.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

TIMING_LABELS = {"TotalTime_s": "total time",
                 "DecisionTime_s": "decision time (go -> leave centre)",
                 "ExecutionTime_s": "execution time (leave centre -> target)",
                 "TakeoffTime_s": "movement takeoff (go -> speed > 5% of peak)"}
spread_labels = globals().get("spread_labels") or (lambda ys, g: np.asarray(ys, float))
TIMING_SHORT = globals().get("TIMING_SHORT") or {
    "TotalTime_s": "total", "DecisionTime_s": "decision",
    "ExecutionTime_s": "execution", "TakeoffTime_s": "takeoff"}
TIMING_MIN_TRIALS = 20


def timing_measures_available(df, measures=TIMING_MEASURES):
    out = [m for m in measures if m in df.columns
           and pd.to_numeric(df[m], errors="coerce").notna().any()]
    miss = [m for m in measures if m not in out]
    if miss:
        print(f"Timing measures with no data in this window, skipped: {miss}")
    return out


def _timed_trials(df, measures):
    """Trials that reached a target, with the timing columns numeric and a
    difficulty axis (distance to the DESIGN boundaries, as in 2.3)."""
    d = df.copy()
    if "ChosenTarget" in d.columns:
        d = d[pd.to_numeric(d["ChosenTarget"], errors="coerce") >= 1]
    for m in measures:
        d[m] = pd.to_numeric(d[m], errors="coerce")
    d["difficulty"] = boundary_distance(d, NOMINAL_BOUNDARIES)
    return d


def takeoff_vs_decision(df):
    """How the kinematic takeoff relates to the task's own decision clock,
    session by session. The gap (DecisionTime - Takeoff) is the time the hand
    is already moving before it crosses the centre circle; a gap that changes
    across sessions means the two clocks are NOT interchangeable and the
    takeoff has to be analysed in its own right."""
    if not {"TakeoffTime_s", "DecisionTime_s"} <= set(df.columns):
        print("TakeoffTime_s or DecisionTime_s missing; takeoff-vs-decision check skipped.")
        return pd.DataFrame()
    d = _timed_trials(df, ["TakeoffTime_s", "DecisionTime_s"])
    d = d.dropna(subset=["TakeoffTime_s", "DecisionTime_s"])
    rows = []
    for s, g in d.groupby("Session", sort=False):
        gap = g["DecisionTime_s"] - g["TakeoffTime_s"]
        rho, p = (stats.spearmanr(g["TakeoffTime_s"], g["DecisionTime_s"])
                  if len(g) >= 10 else (np.nan, np.nan))
        rows.append({"Session": s, "index": int(g["SessionIndex"].iloc[0]), "n": len(g),
                     "median_takeoff_s": g["TakeoffTime_s"].median(),
                     "median_decision_s": g["DecisionTime_s"].median(),
                     "median_gap_s": gap.median(),
                     "pct_takeoff_before_decision": 100.0 * float((gap > 0).mean()),
                     "spearman_rho": rho, "p": p})
    tab = pd.DataFrame(rows).sort_values("index").reset_index(drop=True)
    print("Movement takeoff against the task's decision clock, per session:\n")
    print(tab.round(4).to_string(index=False))
    if len(tab) >= 4:
        rho, p = stats.spearmanr(tab["index"], tab["median_gap_s"])
        print(f"\n  Gap (decision - takeoff) across sessions: Spearman rho = {rho:+.3f}, p = {p:.3g}. "
              f"A gap that shrinks is a hand that crosses the centre sooner after it starts.")
    return tab


def chronometric_by_session(df, measures):
    """Chronometric relation per session and per measure: Spearman rho of the
    measure against distance to the nearest design boundary. A NEGATIVE rho is
    the classic slowing on ambiguous lengths (1.6); doing it per session shows
    whether that effect is stable, and whether it lives in the takeoff (the
    decision is slow to start) or only in the execution (the reach itself)."""
    d = _timed_trials(df, measures)
    rows = []
    for s, g in d.groupby("Session", sort=False):
        for m in measures:
            sub = g[["difficulty", m]].dropna()
            if len(sub) < TIMING_MIN_TRIALS or sub["difficulty"].nunique() < 3:
                rows.append({"Session": s, "index": int(g["SessionIndex"].iloc[0]),
                             "measure": m, "n": len(sub), "rho": np.nan, "p": np.nan})
                continue
            rho, p = stats.spearmanr(sub["difficulty"], sub[m])
            rows.append({"Session": s, "index": int(g["SessionIndex"].iloc[0]),
                         "measure": m, "n": len(sub), "rho": rho, "p": p,
                         "median_s": sub[m].median()})
    per = pd.DataFrame(rows).sort_values(["measure", "index"]).reset_index(drop=True)
    pooled = []
    for m in measures:
        sub = d[["difficulty", m]].dropna()
        rho, p = stats.spearmanr(sub["difficulty"], sub[m]) if len(sub) >= 10 else (np.nan, np.nan)
        per_m = per[per["measure"] == m].dropna(subset=["rho"])
        pooled.append({"measure": m, "n_trials": len(sub), "pooled_rho": rho, "pooled_p": p,
                       "n_sessions": len(per_m),
                       "sessions_negative": int((per_m["rho"] < 0).sum()),
                       "median_session_rho": per_m["rho"].median()})
    pooled = pd.DataFrame(pooled)
    print("\nChronometric relation (timing vs distance to the nearest design boundary):\n")
    print(pooled.round(4).to_string(index=False))
    print("\n  rho < 0 = slower on ambiguous lengths. sessions_negative counts how many")
    print("  sessions show that sign; an effect present in the pooled data but in few")
    print("  sessions is being carried by a subset of days.")
    return per, pooled


def plot_chronometric_by_session(df, per, measures, labels, palette=None):
    """Top: each session is a thin grey line, the pooled median the one dark
    line, so the shape of the chronometric function is readable without
    eleven colours. Bottom: per-session rho, grey when not significant and
    the accent when it is."""
    d = _timed_trials(df, measures)
    n = len(measures)
    fig, axes = plt.subplots(2, n, figsize=(4.6 * n, 7.4), squeeze=False)
    for j, m in enumerate(measures):
        ax = axes[0][j]
        for lab in labels:
            g = d[d["SessionLabel"] == lab].groupby(STIMULUS_COL)[m].agg(["median", "count"])
            g = g[g["count"] >= 3]
            if not g.empty:
                ax.plot(g.index, g["median"], "-", lw=0.9, color=GRAY_LIGHT,
                        label="one session" if lab == labels[0] else None)
        pooled = d.groupby(STIMULUS_COL)[m].median()
        ax.plot(pooled.index, pooled.to_numpy(), "o-", lw=2, ms=4, color=ACCENT,
                label="all sessions, median")
        for b in NOMINAL_BOUNDARIES:
            ax.axvline(b, lw=1, color=GRAY_LIGHT)
        ax.text(NOMINAL_BOUNDARIES[0], ax.get_ylim()[1], "design boundaries",
                fontsize=7, color=GRAY, ha="left", va="top")
        ax.set_xlabel(f"{STIMULUS_COL} (deg VA)")
        ax.set_ylabel("median (s)")
        ax.set_title(TIMING_LABELS.get(m, m), fontsize=9)
        if j == 0:
            ax.legend(fontsize=7, loc="upper right")
        ax = axes[1][j]
        sub = per[per["measure"] == m]
        ax.axhline(0, color=GRAY_LIGHT, lw=1)
        ax.plot(sub["index"], sub["rho"], "-", lw=1, color=GRAY_LIGHT, zorder=1)
        is_sig = sub["p"].notna() & (sub["p"] < 0.05)
        ax.scatter(sub.loc[~is_sig, "index"], sub.loc[~is_sig, "rho"], s=28, color=GRAY, zorder=2)
        ax.scatter(sub.loc[is_sig, "index"], sub.loc[is_sig, "rho"], s=34, color=ACCENT, zorder=3)
        ax.set_xlabel("session (date order)")
        ax.set_ylabel("Spearman rho vs difficulty")
        ax.set_title("per-session rho (colour = p < 0.05)\nbelow 0 = slower near a boundary", fontsize=8)
        ax.set_ylim(-0.6, 0.6)
    fig.suptitle("Timing measures against stimulus length, session by session", y=1.0, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("chronometric_by_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def timing_gee(df, measures):
    """Trial-level model of each timing measure with session as the grouping
    factor, the timing counterpart of the accuracy model in 2.3:

        log(measure) ~ difficulty + session_c     (GEE, Gaussian, exchangeable)

    On the log scale the coefficients read as multiplicative changes: exp(b)
    is the factor per extra deg VA from the boundary (difficulty) and per
    additional session (session_c). Standard errors account for trials being
    correlated within a session."""
    d = _timed_trials(df, measures)
    d["session_c"] = d["SessionIndex"].astype(float) - d["SessionIndex"].astype(float).mean()
    rows = []
    for m in measures:
        sub = d[[m, "difficulty", "session_c", "Session"]].dropna()
        sub = sub[sub[m] > 0]
        if sub["Session"].nunique() < 3 or len(sub) < 100:
            rows.append({"measure": m, "n": len(sub), "note": "too few sessions or trials"})
            continue
        sub = sub.assign(log_t=np.log(sub[m]))
        try:
            fit = smf.gee("log_t ~ difficulty + session_c", groups="Session", data=sub,
                          family=sm.families.Gaussian(),
                          cov_struct=sm.cov_struct.Exchangeable()).fit()
        except Exception as exc:
            rows.append({"measure": m, "n": len(sub), "note": f"fit failed: {type(exc).__name__}"})
            continue
        rows.append({"measure": m, "n": len(sub),
                     "pct_per_deg_from_boundary": 100.0 * (np.exp(fit.params["difficulty"]) - 1.0),
                     "p_difficulty": fit.pvalues["difficulty"],
                     "pct_per_session": 100.0 * (np.exp(fit.params["session_c"]) - 1.0),
                     "p_session": fit.pvalues["session_c"], "note": ""})
    res = pd.DataFrame(rows)
    print("\nGEE per timing measure, log(measure) ~ difficulty + session, grouped by session:\n")
    print(res.round(4).to_string(index=False))
    print("\n  pct_per_deg_from_boundary < 0 = faster on prototypical lengths (chronometric effect);")
    print("  pct_per_session < 0 = the measure shortens from one session to the next (practice).")
    return res


def post_error_by_session(df, measures):
    """Post-error slowing (1.7) repeated per session and per measure: median
    after an error minus median after a correct trial, predecessor taken
    WITHIN the block so block seams are not counted as pairs."""
    d = df.copy()
    d["IsCorrect"] = pd.to_numeric(d["IsCorrect"], errors="coerce")
    d = d.sort_values(["Session", "Block", "TrialNumInBlock", "Attempt"], kind="stable")
    d["prev_correct"] = d.groupby(["Session", "Block"])["IsCorrect"].shift(1)
    d = _timed_trials(d, measures).dropna(subset=["prev_correct"])
    rows = []
    for s, g in d.groupby("Session", sort=False):
        for m in measures:
            a = g.loc[(g["prev_correct"] == 0), m].dropna()
            b = g.loc[(g["prev_correct"] == 1), m].dropna()
            rec = {"Session": s, "index": int(g["SessionIndex"].iloc[0]), "measure": m,
                   "n_after_error": len(a), "n_after_correct": len(b),
                   "delta_s": np.nan, "p": np.nan}
            if len(a) >= 5 and len(b) >= 5:
                rec["delta_s"] = float(a.median() - b.median())
                rec["p"] = float(stats.mannwhitneyu(a, b, alternative="two-sided")[1])
            rows.append(rec)
    res = pd.DataFrame(rows).sort_values(["measure", "index"]).reset_index(drop=True)
    print("\nPost-error slowing per session (median after error - median after correct, s):\n")
    wide = res.pivot(index="index", columns="measure", values="delta_s").reindex(columns=measures)
    print(wide.round(4).to_string())
    for m in measures:
        sub = res[(res["measure"] == m)].dropna(subset=["delta_s"])
        if len(sub):
            print(f"  {m:16s}: slower after an error in {int((sub['delta_s'] > 0).sum())}/{len(sub)} sessions"
                  f" (median delta {sub['delta_s'].median():+.3f} s)")
    return res


EMPHASIS_MEASURE = "TakeoffTime_s"   # the measure this section was added for


def plot_timing_by_session(df, measures, post_error, emphasis=EMPHASIS_MEASURE):
    """Left: median per session for every measure; right: post-error slowing
    per session. One measure carries the accent, the rest are grey context
    with a direct label at the line end instead of a legend."""
    d = _timed_trials(df, measures)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    ax = axes[0]
    x_end = int(d["SessionIndex"].max())
    meds = {m: d.groupby("SessionIndex")[m].median() for m in measures}
    y_all = np.concatenate([g.to_numpy() for g in meds.values()])
    label_y = spread_labels([g.iloc[-1] for g in meds.values()],
                            0.05 * (np.nanmax(y_all) - np.nanmin(y_all)))
    for (m, g), ly in zip(meds.items(), label_y):
        hi = m == emphasis
        ax.plot(g.index, g.to_numpy(), "o-", color=ACCENT if hi else GRAY,
                lw=1.8 if hi else 1.2, ms=4 if hi else 3)
        ax.text(x_end + 0.25, ly, TIMING_SHORT.get(m, m), va="center",
                fontsize=8, color=ACCENT if hi else GRAY)
    ax.set_xlim(0.5, x_end + 2.5)
    ax.set_xlabel("session (date order)")
    ax.set_ylabel("median (s)")
    ax.set_title("Timing measures across sessions", fontsize=10)
    ax = axes[1]
    ax.axhline(0, color=GRAY_LIGHT, lw=1)
    subs = {m: post_error[post_error["measure"] == m].dropna(subset=["delta_s"]) for m in measures}
    subs = {m: s for m, s in subs.items() if len(s)}
    y_all = np.concatenate([s["delta_s"].to_numpy() for s in subs.values()]) if subs else np.array([0.0])
    label_y = spread_labels([s["delta_s"].iloc[-1] for s in subs.values()],
                            0.05 * (np.nanmax(y_all) - np.nanmin(y_all)))
    for (m, sub), ly in zip(subs.items(), label_y):
        hi = m == emphasis
        ax.plot(sub["index"], sub["delta_s"], "o-", color=ACCENT if hi else GRAY,
                lw=1.8 if hi else 1.2, ms=4 if hi else 3)
        ax.text(sub["index"].iloc[-1] + 0.25, ly, TIMING_SHORT.get(m, m),
                va="center", fontsize=8, color=ACCENT if hi else GRAY)
    ax.set_xlim(0.5, x_end + 2.5)
    ax.set_xlabel("session (date order)")
    ax.set_ylabel("after error - after correct (s)")
    ax.set_title("Post-error slowing per session (above 0 = slower after an error)", fontsize=10)
    fig.tight_layout()
    fig.savefig("timing_by_session.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


if MULTI_DF.empty:
    print("No pooled trial table (2.1 found no sessions); nothing to compare.")
else:
    TIMING_AVAILABLE = timing_measures_available(MULTI_DF)
    if "SessionLabel" not in MULTI_DF.columns:
        _stamp = MULTI_DF["SessionStart"].dt.strftime("%d-%b %H:%M").fillna("no date")
        MULTI_DF["SessionLabel"] = MULTI_DF["SessionIndex"].astype(str).str.zfill(2) + " | " + _stamp
    _labels = (MULTI_DF.drop_duplicates("Session").sort_values("SessionIndex")["SessionLabel"].tolist())
    _palette = session_palette(_labels)

    TAKEOFF_VS_DECISION = takeoff_vs_decision(MULTI_DF)
    CHRONO_BY_SESSION, CHRONO_POOLED = chronometric_by_session(MULTI_DF, TIMING_AVAILABLE)
    plot_chronometric_by_session(MULTI_DF, CHRONO_BY_SESSION, TIMING_AVAILABLE, _labels)
    plt.show()
    TIMING_GEE = timing_gee(MULTI_DF, TIMING_AVAILABLE)
    POST_ERROR_BY_SESSION = post_error_by_session(MULTI_DF, TIMING_AVAILABLE)
    plot_timing_by_session(MULTI_DF, TIMING_AVAILABLE, POST_ERROR_BY_SESSION)
    plt.show()
    TAKEOFF_VS_DECISION.to_csv("takeoff_vs_decision_by_session.csv", index=False)
    CHRONO_BY_SESSION.to_csv("chronometric_by_session.csv", index=False)
    TIMING_GEE.to_csv("timing_gee_across_sessions.csv", index=False)
    POST_ERROR_BY_SESSION.to_csv("post_error_by_session.csv", index=False)


### 2.7. Embeddings: PCA and t-SNE by category, by outcome and by session

The per-trial kinematic features of every session (2.4) projected to two
dimensions twice: **PCA**, linear, where distances mean what they mean in the
feature space, and **t-SNE**, non-linear, which pulls local neighbourhoods
into visible clusters at the price of distorting global distances. Each row is
coloured three ways: by the true category (`StimulusGroup`), by outcome
(`IsCorrect`) and by session (viridis, dark = earliest), so a cluster that is
really a session effect is not mistaken for a category one.

`BarSizeVA_deg` is held out of the features by default: the category is a
deterministic function of it, so including it (as 1.3 does) would make the
category clusters a restatement of the design. `EMBED_INCLUDE_STIMULUS`
flips that.

The table under the figure quantifies what the eye sees. The silhouette says
whether a labelling forms clusters (near 0 = it does not), and a k-NN read-out
in the same space says whether the label is nonetheless readable from the
movement. The full feature space is included as the ceiling, since a 2-D
picture can only lose separation.


In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import silhouette_score
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_score, StratifiedKFold
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import silhouette_score
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_score, StratifiedKFold

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Same feature set as 1.3 (hold_* drop out on their own when the movement
# window has no target hold). BarSizeVA_deg is NOT in by default: the category
# is a deterministic function of it, so with it in, "clusters by category"
# would only restate the design. Set EMBED_INCLUDE_STIMULUS = True to
# reproduce the 1.3 choice.
EMBED_FEATURES = ["DecisionTime_s", "ExecutionTime_s", "TakeoffTime_s", "n_samples",
                  "peak_vel_cm", "mean_vel_cm", "peak_accel_cm", "mean_accel_cm",
                  "path_length_cm", "straightness",
                  "hold_dur_ms", "hold_max_speed_cm", "hold_excursion_px"]
EMBED_INCLUDE_STIMULUS = False
EMBED_LABELS = ["StimulusGroup", "IsCorrect", "SessionLabel"]
TSNE_PERPLEXITY = 30
TSNE_PCA_DIMS = 10          # t-SNE runs on the leading PCs, the usual denoising step
TSNE_MAX_TRIALS = 6000      # subsample above this for speed (stratified by category)
EMBED_SEED = 0
KNN_K = 15
PCA_AXIS_PCTL = 0.5        # PCA panels: axis limits at this / (100 - this) percentile


def embedding_matrix(df, features=EMBED_FEATURES, include_stimulus=EMBED_INCLUDE_STIMULUS):
    feats = list(features) + (["BarSizeVA_deg"] if include_stimulus else [])
    d = df.copy()
    keep = []
    for f in feats:
        if f not in d.columns:
            continue
        d[f] = pd.to_numeric(d[f], errors="coerce")
        if d[f].notna().any() and d[f].nunique(dropna=True) > 1:
            keep.append(f)
    skipped = [f for f in feats if f not in keep]
    if skipped:
        print(f"Features skipped (absent, empty or constant): {skipped}")
    fit = d.dropna(subset=keep).reset_index(drop=True)
    print(f"Embedding {len(fit)}/{len(d)} complete trials x {len(keep)} features"
          f"{' (BarSizeVA_deg included)' if include_stimulus else ' (BarSizeVA_deg held out)'}.")
    X = StandardScaler().fit_transform(fit[keep].to_numpy(float))
    return fit, X, keep


def compute_embeddings(X, seed=EMBED_SEED, perplexity=TSNE_PERPLEXITY, pca_dims=TSNE_PCA_DIMS):
    pca = PCA(random_state=seed).fit(X)
    scores = pca.transform(X)
    n_in = min(pca_dims, scores.shape[1])
    tsne = TSNE(n_components=2, perplexity=min(perplexity, max(5, (len(X) - 1) // 3)),
                init="pca", learning_rate="auto", random_state=seed)
    emb = tsne.fit_transform(scores[:, :n_in])
    return pca, scores, emb


def _label_palette(lab, levels, labels_order=None):
    if lab == "StimulusGroup":
        return category_palette(levels)
    if lab == "IsCorrect":
        return correctness_palette(levels)
    if lab == "SessionLabel":
        return session_palette(labels_order or sorted(levels))
    return {lv: plt.get_cmap("tab10")(i % 10) for i, lv in enumerate(levels)}


def plot_embeddings(fit, scores, emb, evr, labels=EMBED_LABELS, session_order=None):
    labels = [l for l in labels if l in fit.columns]
    fig, axes = plt.subplots(2, len(labels), figsize=(5.6 * len(labels), 10.4), squeeze=False)
    for j, lab in enumerate(labels):
        vals = fit[lab].to_numpy()
        levels = [lv for lv in (session_order if lab == "SessionLabel" and session_order
                                else sorted(pd.unique(pd.Series(vals).dropna())))
                  if (vals == lv).any()]
        pal = _label_palette(lab, levels, session_order)
        # minority levels drawn last so they are not buried under the majority
        order = sorted(levels, key=lambda lv: -(vals == lv).sum())
        for row, (Z, xl, yl) in enumerate(((scores[:, :2], f"PC1 ({evr[0]:.1%})", f"PC2 ({evr[1]:.1%})"),
                                          (emb, "t-SNE 1", "t-SNE 2"))):
            ax = axes[row][j]
            for lv in order:
                m = vals == lv
                name = CORRECTNESS_LABELS.get(int(lv), lv) if lab == "IsCorrect" else lv
                ax.scatter(Z[m, 0], Z[m, 1], s=9, alpha=0.55, edgecolor="none",
                           color=pal.get(lv, "#999999"), label=f"{name} (n={int(m.sum())})")
            ax.set_xlabel(xl)
            ax.set_ylabel(yl)
            ax.set_title(f"{'PCA' if row == 0 else 't-SNE'}, coloured by {lab}", fontsize=10)
            if lab == "SessionLabel":
                ax.legend(fontsize=5, ncol=2, frameon=False, markerscale=2)
            else:
                ax.legend(fontsize=8, frameon=False, markerscale=2)
            if row == 0:
                ax.axhline(0, lw=0.6, color=GRAY_LIGHT)
                ax.axvline(0, lw=0.6, color=GRAY_LIGHT)
                # a handful of extreme trials would otherwise squash the cloud
                # into a corner; the points are still there, only off-axis
                lo, hi = np.percentile(Z, [PCA_AXIS_PCTL, 100 - PCA_AXIS_PCTL], axis=0)
                pad = 0.05 * (hi - lo)
                ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
                ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
                ax.set_title(f"PCA, coloured by {lab}\n(axes clipped to the "
                             f"{PCA_AXIS_PCTL:g}-{100 - PCA_AXIS_PCTL:g} percentile)", fontsize=10)
            else:
                ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("Per-trial kinematic features, all sessions: PCA (top) and t-SNE (bottom)",
                 y=1.0, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig("embeddings_pca_tsne.png", dpi=FIGURE_DPI, bbox_inches="tight")
    return fig


def separation_table(fit, X, scores, emb, labels=EMBED_LABELS, seed=EMBED_SEED, k=KNN_K):
    """How separable is each labelling, in each space?

    silhouette : -1..1, how much closer a trial sits to its own label than
                 to the nearest other label. Near 0 = the labels do not form
                 clusters in that space, even if a classifier can still
                 separate them.
    knn_acc    : balanced cross-validated accuracy of a k-NN read-out in the
                 same space, against the chance level for that label's class
                 balance. Reported for the 2-D embeddings so the picture and
                 the number describe the same thing, and for the full
                 standardised feature space as the ceiling.
    """
    rows = []
    spaces = {"features (all dims)": X, "PCA (2-D)": scores[:, :2], "t-SNE (2-D)": emb}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for lab in [l for l in labels if l in fit.columns]:
        y = fit[lab].astype(str).to_numpy()
        counts = pd.Series(y).value_counts()
        if counts.size < 2 or counts.min() < 10:
            continue
        chance = float((counts / counts.sum()).max())
        for name, Z in spaces.items():
            sil = silhouette_score(Z, y, random_state=seed) if len(y) > counts.size else np.nan
            knn = KNeighborsClassifier(n_neighbors=k)
            acc = cross_val_score(knn, Z, y, cv=cv, scoring="balanced_accuracy").mean()
            rows.append({"label": lab, "space": name, "n_levels": int(counts.size),
                         "silhouette": sil, "knn_balanced_acc": acc,
                         "chance": 1.0 / counts.size})
    res = pd.DataFrame(rows)
    print("\nSeparation of each labelling in each space:\n")
    print(res.round(3).to_string(index=False))
    print("\n  silhouette near 0 = no clusters by that label; knn_balanced_acc above chance")
    print("  = the label is still readable from the movement even without clusters.")
    return res


if "MULTI_ANALYSIS_DF" not in globals() or MULTI_ANALYSIS_DF.empty:
    print("No pooled feature table (run 2.4 first); embeddings skipped.")
else:
    EMBED_FIT, EMBED_X, EMBED_COLS = embedding_matrix(MULTI_ANALYSIS_DF)
    if len(EMBED_FIT) > TSNE_MAX_TRIALS:
        _rng = np.random.default_rng(EMBED_SEED)
        _idx = (EMBED_FIT.groupby("StimulusGroup", group_keys=False)
                .apply(lambda g: g.sample(frac=TSNE_MAX_TRIALS / len(EMBED_FIT), random_state=EMBED_SEED))
                .index.to_numpy())
        EMBED_FIT, EMBED_X = EMBED_FIT.loc[_idx].reset_index(drop=True), EMBED_X[_idx]
        print(f"Subsampled to {len(EMBED_FIT)} trials for t-SNE (stratified by category).")
    EMBED_PCA, EMBED_SCORES, EMBED_TSNE = compute_embeddings(EMBED_X)
    _evr = EMBED_PCA.explained_variance_ratio_
    print(f"PCA: PC1 {_evr[0]:.1%}, PC2 {_evr[1]:.1%} of the variance "
          f"({int(np.searchsorted(np.cumsum(_evr), 0.9) + 1)} components reach 90%).")
    _load = pd.DataFrame(EMBED_PCA.components_[:2].T, index=EMBED_COLS, columns=["PC1", "PC2"])
    for pc in ("PC1", "PC2"):
        top = _load[pc].abs().sort_values(ascending=False).index[:3]
        print(f"  {pc}: " + ", ".join(f"{f} ({_load.loc[f, pc]:+.2f})" for f in top))
    plot_embeddings(EMBED_FIT, EMBED_SCORES, EMBED_TSNE, _evr, session_order=SESSION_LABELS)
    plt.show()
    EMBED_SEPARATION = separation_table(EMBED_FIT, EMBED_X, EMBED_SCORES, EMBED_TSNE)
    EMBED_SEPARATION.to_csv("embedding_separation.csv", index=False)
    _out = EMBED_FIT[[c for c in ["Session", "SessionIndex", "Block", "TrialNumInBlock", "Attempt",
                                  "StimulusGroup", "BarSizeVA_deg", "IsCorrect", "ChosenTarget"]
                      if c in EMBED_FIT.columns]].copy()
    _out["PC1"], _out["PC2"] = EMBED_SCORES[:, 0], EMBED_SCORES[:, 1]
    _out["tsne1"], _out["tsne2"] = EMBED_TSNE[:, 0], EMBED_TSNE[:, 1]
    _out.to_csv("embeddings_pca_tsne.csv", index=False)
